In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date
from collections import defaultdict

# =============================================================
#  Smart APS V12-DEMAND  —  Demand-First, Protected Planning
# =============================================================
# KEY FIXES vs V11:
#  1. Demand is TOPMOST priority — parts with demand=0 inventory
#     are scheduled FIRST before any inventory building
#  2. No part can be starved of its demand just because a machine
#     is busy making extra inventory for another part
#  3. Max 3 parts per machine (was 4)
#  4. Per-part decision sheet added (VT_Part_Decision)
#  5. Daily totals sheet: total production qty + total changeovers
#  6. Demand met = inventory_before + produced_today >= demand_daily
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 4, 9)
INDENT_MONTH  = date(2026, 4, 1)

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS          = 22
AVAILABLE_HOURS_EXTENDED = 23
MIN_RUN_HOURS            = 4
MACHINE_STATE_FILE       = "machine_state.json"

MIN_DAILY_INDENT         = 150
MIN_INDENT_HOURS         = 4.0

SAFETY_DAYS              = 3
TARGET_DAYS              = 5

OPD_SCENARIO_0 = 3.0
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT  = 90.0
COLOR_PURGE_HRS  = 10 / 60.0

RUNNER_PRIORITY_DAYS = 2.0

# FIX: Max 3 parts per machine (was 4 before)
MAX_PARTS_PER_MACHINE    = 3

MAX_DAILY_CO             = 25
FORWARD_LOOK_DAYS        = 7

TERMINAL_THRESHOLD = {
    "Runner":   1.0,
    "Repeater": 0.5,
    "Stranger": 0.25,
}

TERMINAL_RELAXATION_PCT  = 0.15

STRATEGIC_BUFFER_DAYS        = 7
STRATEGIC_PRIORITY_DISCOUNT  = 0.5
EFFICIENCY_MODE_UTIL_FLOOR   = 95.0
ABSOLUTE_MAX_DAYS            = 15

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
terminal_path   = "C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx"
demand_path     = "C:/Users/Ex0164/Demand.xlsx"
output_path     = f"Smart_APS_V12_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

DEMAND_SHEET_NAME  = "VT_Demand"
DEMAND_PART_COL    = "Part"
DEMAND_DAILY_COL   = "Daily_Demand"

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V12-DEMAND  —  Demand-First, Protected Planning")
print(f"  Planning date  : {PLANNING_DATE}")
print(f"  Indent month   : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days   : {WORKING_DAYS}  ({TOTAL_DAYS} days - {SUNDAY_COUNT} Sundays)")
print(f"  Safety floor   : {SAFETY_DAYS} days  |  Target ceiling : {TARGET_DAYS} days")
print(f"  Max parts/machine: {MAX_PARTS_PER_MACHINE}")
print(f"  Max daily CO   : {MAX_DAILY_CO}")
print(f"  Terminal relax : {int(TERMINAL_RELAXATION_PCT*100)}%")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw         = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix            = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw            = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path,     sheet_name="VT_Machine_Part_Count")

try:
    vt_fixed_raw = pd.read_excel(matrix_path, sheet_name="VT_Fixed")
    print(f"  VT_Fixed sheet loaded  ({len(vt_fixed_raw)} rows)")
except Exception as _fe:
    vt_fixed_raw = None
    print(f"  WARNING: VT_Fixed sheet not found ({_fe})")

try:
    vt_terminals_raw      = pd.read_excel(terminal_path, sheet_name="VT_Terminals")
    vt_terminal_avail_raw = pd.read_excel(terminal_path, sheet_name="VT_Terminal_Inventory")
    print(f"  Terminal data loaded")
except FileNotFoundError:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: terminal_path not found — terminal constraint DISABLED.")
except Exception as _te:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: Could not load terminal data ({_te})")

demand_monthly_raw = {}
demand_daily_raw   = {}

try:
    demand_df = pd.read_excel(demand_path, sheet_name=DEMAND_SHEET_NAME)
    demand_df.columns = [str(c).strip().lstrip('\ufeff') for c in demand_df.columns]
    print(f"  Demand file columns: {list(demand_df.columns)}")

    _part_col_d = next(
        (c for c in demand_df.columns if c.strip().lower() == DEMAND_PART_COL.lower()), None
    )
    _dem_col_d = next(
        (c for c in demand_df.columns if c.strip().lower() == DEMAND_DAILY_COL.lower()), None
    )
    if _part_col_d is None:
        _part_col_d = next((c for c in demand_df.columns if 'part' in c.strip().lower()), None)
    if _dem_col_d is None:
        _dem_col_d = next(
            (c for c in demand_df.columns if 'daily' in c.strip().lower() or 'demand' in c.strip().lower()), None
        )

    if _part_col_d and _dem_col_d:
        for _, row in demand_df.iterrows():
            p = row[_part_col_d]
            v = row[_dem_col_d]
            if pd.isna(p) or str(p).strip() == "":
                continue
            part_key = str(p).strip()
            try:
                daily_dem = float(v) if pd.notna(v) else 0.0
            except (ValueError, TypeError):
                daily_dem = 0.0
            demand_daily_raw[part_key]   = round(daily_dem, 4)
            demand_monthly_raw[part_key] = round(daily_dem * WORKING_DAYS, 4)
        print(f"  Demand loaded: {len(demand_daily_raw)} parts")
    else:
        print(f"  WARNING: Demand columns not found — demand constraint DISABLED.")
except FileNotFoundError:
    print(f"  WARNING: demand_path not found — demand constraint DISABLED.")
except Exception as _de:
    print(f"  WARNING: Could not load demand data ({_de})")

# =============================================================
# SECTION 5A — EFFECTIVE DAILY / MONTHLY HELPERS
# =============================================================

def effective_daily(part: str) -> float:
    ind = indent_daily.get(part, 0.0)
    dem = demand_daily_raw.get(part, 0.0)
    return max(ind, dem)

def effective_monthly(part: str) -> float:
    ind = indent_monthly.get(part, 0.0)
    dem = demand_monthly_raw.get(part, 0.0)
    return max(ind, dem)

def demand_driver(part: str) -> str:
    ind = indent_daily.get(part, 0.0)
    dem = demand_daily_raw.get(part, 0.0)
    if dem > ind + 0.001:
        return f"DEMAND({dem:.2f}>indent {ind:.2f})"
    elif ind > dem + 0.001:
        return f"INDENT({ind:.2f})"
    elif ind > 0:
        return f"EQUAL({ind:.2f})"
    else:
        return "NO_DRIVER"

def demand_today_qty(part: str, current_inv: float) -> float:
    """How much we must produce TODAY to cover today's demand gap."""
    dem = demand_daily_raw.get(part, 0.0)
    if dem <= 0:
        return 0.0
    gap = dem - current_inv
    return max(0.0, gap)

def is_demand_met(part: str, inv_before: float, produced: float) -> bool:
    """Demand is met if inventory_before + produced >= demand_daily."""
    dem = demand_daily_raw.get(part, 0.0)
    if dem <= 0:
        return True
    return (inv_before + produced) >= (dem - 0.5)

# =============================================================
# SECTION 6 — PARSE VT SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(f"Column '{name}' not found in sheet '{sheet}'.\nAvailable: {list(df.columns)}")
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")
vt_col_color     = find_col(vt_parts_raw, "Color",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()
data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]
data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet: {len(data)}  |  With valid rate: {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)
indent_daily   = {p: round(qty / WORKING_DAYS, 4) for p, qty in indent_monthly.items()}

tools_available = {}
part_color      = {}

for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1
    c = row[vt_col_color]
    part_color[p] = str(c).strip().upper() if pd.notna(c) and str(c).strip() not in ("", "nan") else "UNKNOWN"

ALL_KNOWN_COLORS = dict(part_color)

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours: {len(color_groups)}")

today_target_qty = {
    p: max(0.0, effective_daily(p) - inventory.get(p, 0.0))
    for p in set(list(indent_monthly.keys()) + list(demand_daily_raw.keys()))
}

# =============================================================
# SECTION 7A — FIXED MACHINE CONSTRAINT
# =============================================================

def build_fixed_machine_dicts(df):
    pfm, mfp = {}, {}
    if df is None or df.empty:
        return pfm, mfp
    machine_col = next((c for c in df.columns if str(c).strip().lower() == "machine"), None)
    if machine_col is None:
        return pfm, mfp
    part_cols = [c for c in df.columns if str(c).strip().lower() != "machine"]
    for _, row in df.iterrows():
        machine = row[machine_col]
        if pd.isna(machine) or str(machine).strip() == "":
            continue
        m = str(machine).strip()
        for col in part_cols:
            val = row[col]
            if pd.isna(val) or str(val).strip() in ("", "nan"):
                continue
            p = str(val).strip()
            if p in pfm:
                continue
            pfm[p] = m
            mfp.setdefault(m, []).append(p)
    return pfm, mfp

part_fixed_machine, machine_fixed_parts = build_fixed_machine_dicts(vt_fixed_raw)
print(f"  Fixed machine mappings: {len(part_fixed_machine)} parts")

# =============================================================
# SECTION 7B — TERMINAL CONSTRAINT
# =============================================================

def _build_part_terminals(df):
    result = {}
    if df is None or df.empty:
        return result
    part_col = next((c for c in df.columns if str(c).strip().lower() in ("part", "material")), None)
    if part_col is None:
        return result
    terminal_cols = [c for c in df.columns if str(c).strip().lower() not in ("part", "material")]
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        p = str(part).strip()
        terminals = [str(row[col]).strip().upper() for col in terminal_cols
                     if pd.notna(row[col]) and str(row[col]).strip() not in ("", "nan")]
        if terminals:
            result[p] = terminals
    return result

def _build_terminal_status(df):
    result = {}
    if df is None or df.empty:
        return result
    term_col = next((c for c in df.columns if str(c).strip().lower() == "terminal"), None)
    inv_col  = next((c for c in df.columns if str(c).strip().lower() == "inventory"), None)
    if term_col is None or inv_col is None:
        return result
    for _, row in df.iterrows():
        t = row[term_col]
        v = row[inv_col]
        if pd.isna(t) or str(t).strip() == "":
            continue
        key = str(t).strip().upper()
        try:
            qty = float(v) if pd.notna(v) else 0.0
        except (ValueError, TypeError):
            qty = 0.0
        result[key] = qty
    return result

part_terminals  = _build_part_terminals(vt_terminals_raw)
terminal_status = _build_terminal_status(vt_terminal_avail_raw)

if terminal_status:
    zero_count  = sum(1 for v in terminal_status.values() if v <= 0)
    print(f"  Terminals: {len(terminal_status)} total | {zero_count} at ZERO")


def terminal_blocked(part, category_override=None):
    """Returns (blocked: bool, reason: str, relaxed: bool)"""
    required = part_terminals.get(part, [])
    if not required:
        return False, "", False

    cat   = category_override or part_category.get(part, "Stranger")
    daily = effective_daily(part)
    mult  = TERMINAL_THRESHOLD.get(cat, 0.25)
    strict_threshold  = mult * daily
    relaxed_threshold = strict_threshold * (1.0 - TERMINAL_RELAXATION_PCT)

    hard_blocking = []
    soft_blocking = []

    for t in required:
        t_inv = terminal_status.get(t, 0)
        if t_inv < relaxed_threshold:
            hard_blocking.append(
                f"{t}(inv={t_inv:.0f} < relaxed_need={relaxed_threshold:.0f})"
            )
        elif t_inv < strict_threshold:
            soft_blocking.append(
                f"{t}(inv={t_inv:.0f} ~ within {int(TERMINAL_RELAXATION_PCT*100)}% of need={strict_threshold:.0f})"
            )

    if hard_blocking:
        return True, f"Terminal HARD BLOCK: {', '.join(hard_blocking)}", False
    if soft_blocking:
        return False, f"Terminal RELAXED: {', '.join(soft_blocking)}", True
    return False, "", False


def terminal_coverage_ratio(part):
    required = part_terminals.get(part, [])
    if not required:
        return 1.0
    cat   = part_category.get(part, "Stranger")
    daily = effective_daily(part)
    mult  = TERMINAL_THRESHOLD.get(cat, 0.25)
    threshold = mult * daily
    if threshold <= 0:
        return 1.0
    ratios = [min(1.0, terminal_status.get(t, 0) / threshold) for t in required]
    return round(min(ratios), 4)

# =============================================================
# SECTION 7C — SKIP / ELIGIBILITY RULES
# =============================================================

# FIX: A part with demand > 0 and zero inventory is NEVER skipped.
# It must enter the scheduler regardless of indent thresholds.
def should_skip(part):
    """
    Returns (skip: bool, reason: str)
    DEMAND-PROTECTED: Parts with demand > 0 bypass indent thresholds.
    """
    ind_daily  = indent_daily.get(part, 0.0)
    dem_daily  = demand_daily_raw.get(part, 0.0)
    eff_daily  = effective_daily(part)
    monthly    = indent_monthly.get(part, 0.0)
    r          = rate.get(part, 1.0)
    inv        = inventory.get(part, 0.0)

    has_demand = dem_daily > 0.0
    has_demand_gap = has_demand and inv < dem_daily  # Today's demand not covered by existing stock

    # If today's demand is not covered by current stock, NEVER skip.
    if has_demand_gap:
        # Only terminal hard block can skip it
        t_blocked, t_reason, _ = terminal_blocked(part)
        if t_blocked:
            return True, t_reason
        return False, ""

    # Parts with demand but stock already covers it — still allow scheduling (for inventory build)
    # but can be ceiling-skipped
    if not has_demand:
        if ind_daily <= MIN_DAILY_INDENT:
            return True, f"Daily indent {ind_daily:.2f} <= {MIN_DAILY_INDENT} threshold (no demand)"
        indent_hrs = monthly / r if r > 0 else 0.0
        if indent_hrs <= MIN_INDENT_HOURS:
            return True, f"Monthly indent = {indent_hrs:.2f}h <= {MIN_INDENT_HOURS}h threshold (no demand)"

    # Inventory ceiling check — only skip if well above target
    if eff_daily > 0 and inv >= TARGET_DAYS * eff_daily:
        # But if demand is not met, don't skip
        if has_demand and not is_demand_met(part, inv, 0):
            return False, ""
        return True, (
            f"Inventory ({inv:.0f}) >= {TARGET_DAYS}-day target "
            f"({TARGET_DAYS * eff_daily:.0f} pcs) — at ceiling"
        )

    t_blocked, t_reason, _ = terminal_blocked(part)
    if t_blocked:
        return True, t_reason

    return False, ""


def is_hard_skip(part):
    t_blocked, _, _ = terminal_blocked(part)
    return t_blocked

# =============================================================
# SECTION 7D — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7E — MACHINE PART COUNT
# =============================================================

def build_machine_part_count(df):
    mpc = {}
    machine_col = next((c for c in df.columns if str(c).strip().lower() == "machine"), None)
    count_col   = next((c for c in df.columns if str(c).strip().lower() == "part_count"), None)
    if machine_col is None or count_col is None:
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1

# =============================================================
# SECTION 7F — PART CATEGORY
# =============================================================

def build_category(df):
    cat = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}
for p in demand_daily_raw:
    if p not in part_category:
        part_category[p] = "Stranger"

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                os.remove(MACHINE_STATE_FILE)
                return {}
            unknown = [p for p in state.values() if p not in part_color]
            for p in unknown:
                ALL_KNOWN_COLORS[p] = "NEEDS_PURGE"
            print(f"  Machine state loaded ({len(state)} machines)")
            return state
        except Exception:
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state: FIRST RUN")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved -> '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

machine_compatible_parts = {m: [] for m in vt_machines}
for p, machines in vt_compat.items():
    for m in machines:
        if m in machine_compatible_parts:
            machine_compatible_parts[m].append(p)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = effective_daily(p)
        skip, _ = should_skip(p)
        if daily == 0:
            continue
        # Don't skip zero-inv parts from scenario classification
        coverage.append(inv / daily)
    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"
    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < SAFETY_DAYS)
    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {SAFETY_DAYS}-day safety floor"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (>={SAFETY_DAYS} days)"

# =============================================================
# SECTION 11 — OPD CAP
# =============================================================

def opd_cap(scenario_id):
    return min({0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1, 2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2), TARGET_DAYS)

def effective_opd_cap_qty(part, scenario_id, current_inv):
    eff_d  = effective_daily(part)
    dem_d  = demand_daily_raw.get(part, 0.0)
    cap_q  = opd_cap(scenario_id) * eff_d
    # Demand floor: always allow at least enough to meet today's demand
    demand_floor_qty = max(0.0, dem_d - current_inv) if dem_d > 0 else 0.0
    return max(cap_q, demand_floor_qty)

# =============================================================
# SECTION 12 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = effective_daily(p)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_score    = min(1.0, max(0.0, (TARGET_DAYS - days_cov) / TARGET_DAYS))
        velocity_raw = daily / max(float(inv), 1.0) if daily > 0 else 0.0
        # FIX: Boost priority for zero-inventory + demand parts
        has_demand_gap = demand_daily_raw.get(p, 0) > 0 and inv < demand_daily_raw.get(p, 0)
        rows.append({
            "part": p, "inv": inv, "daily": daily, "days_cov": days_cov, "cat": cat,
            "gap_score": gap_score, "velocity_raw": velocity_raw,
            "has_demand_gap": has_demand_gap,
        })
    if not rows:
        return {}, []
    max_daily    = max(r["daily"] for r in rows) or 1.0
    max_velocity = max(r["velocity_raw"] for r in rows) or 1.0
    scores, score_rows = {}, []
    for r in rows:
        p = r["part"]
        gap_pct      = r["gap_score"] * 100.0
        velocity_pct = (r["velocity_raw"] / max_velocity) * 100.0
        urgency_score  = 0.60 * gap_pct + 0.40 * velocity_pct
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100.0
        final_score = (
            W_URGENCY  * urgency_score +
            W_CATEGORY * category_score +
            W_INDENT   * indent_score
        )
        # FIX: Hard-boost parts with demand gap (zero inv + demand)
        if r["has_demand_gap"]:
            final_score += 200.0
        scores[p] = round(final_score, 2)
        _, t_reason, t_relaxed = terminal_blocked(p)
        inv_today = inventory.get(p, 0)
        score_rows.append({
            "Part": p, "Category": r["cat"],
            "Color": part_color.get(p, "UNKNOWN"),
            "Fixed_Machine": part_fixed_machine.get(p, "—"),
            "Tools": tools_available.get(p, 1),
            "Inventory_Now": round(r["inv"], 0),
            "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
            "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
            "Effective_Daily": round(r["daily"], 2),
            "Demand_Driver": demand_driver(p),
            "Demand_Gap_Today": round(max(0.0, demand_daily_raw.get(p, 0.0) - inv_today), 2),
            "Demand_Covered_By_Stock": "YES" if inv_today >= demand_daily_raw.get(p, 0.0) else "NO",
            "Terminal_Coverage_Ratio": terminal_coverage_ratio(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Days_Coverage": round(r["days_cov"], 2),
            "Buffer_Status": (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "BUILDING"     if r["days_cov"] < TARGET_DAYS else
                "AT_TARGET"
            ),
            "Final_Score": round(final_score, 2),
            "Demand_Boost_Applied": "YES" if r["has_demand_gap"] else "No",
        })
    return scores, score_rows

# =============================================================
# SECTION 13 — COLOUR-AWARE CHANGEOVER HELPER
# =============================================================

def _co_hrs_for(part, machine, machine_last_part):
    last = machine_last_part.get(machine)
    if last is None or last == part:
        return 0.0
    base_co    = vt_changeover.get(machine, DEFAULT_CHANGEOVER_HRS)
    last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
    new_color  = part_color.get(part, "UNKNOWN")
    if last_color == "NEEDS_PURGE":
        return base_co + COLOR_PURGE_HRS
    purge = (
        COLOR_PURGE_HRS
        if last_color != new_color and last_color not in ("UNKNOWN",) and new_color not in ("UNKNOWN",)
        else 0.0
    )
    return base_co + purge

# =============================================================
# SECTION 14 — MACHINE PART COUNT GUARD
# =============================================================

def parts_on_machine(machine, plan):
    """Count distinct parts already assigned to this machine."""
    return len({r["Part"] for r in plan if r["Machine"] == machine})

def machine_has_capacity_for_new_part(machine, part, plan):
    """
    FIX: Enforce MAX_PARTS_PER_MACHINE = 3.
    If part is already on this machine, it's an extension (OK).
    If it's a new part, check we haven't hit the 3-part limit.
    """
    parts_already = {r["Part"] for r in plan if r["Machine"] == machine}
    if part in parts_already:
        return True  # Extension of existing part — always OK
    return len(parts_already) < MAX_PARTS_PER_MACHINE

# =============================================================
# SECTION 14A — MACHINE RANKER
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days, plan,
                  exclude_fixed_machines=True,
                  allow_fixed_overflow=False):
    category  = part_category.get(part, "Stranger")
    new_color = part_color.get(part, "UNKNOWN")
    fixed_m   = part_fixed_machine.get(part)
    is_fixed  = fixed_m is not None
    runner_lock = (category == "Runner" and inv_days < RUNNER_PRIORITY_DAYS and not is_fixed)

    effective_candidates = []
    for m in machines_to_try:
        if exclude_fixed_machines and not allow_fixed_overflow and m in machine_fixed_parts:
            if not is_fixed:
                continue
            elif m != fixed_m:
                continue
        effective_candidates.append(m)

    if is_fixed and fixed_m in effective_candidates:
        used_f = machine_hours.get(fixed_m, 0)
        free_f = round(AVAILABLE_HOURS - used_f, 4)
        co_f   = _co_hrs_for(part, fixed_m, machine_last_part)
        eff_f  = round(free_f - co_f, 4)
        if eff_f >= MIN_RUN_HOURS and machine_has_capacity_for_new_part(fixed_m, part, plan):
            fallback = _rank_normal(
                part, [m for m in effective_candidates if m != fixed_m],
                machine_hours, machine_last_part, new_color, runner_lock, plan
            )
            return [(fixed_m, co_f, eff_f, -1.0)] + fallback, runner_lock
        remaining = [m for m in effective_candidates if m != fixed_m]
        return _rank_normal(part, remaining, machine_hours, machine_last_part, new_color, runner_lock, plan), runner_lock

    return _rank_normal(part, effective_candidates, machine_hours, machine_last_part, new_color, runner_lock, plan), runner_lock


def _rank_normal(part, machines_to_try, machine_hours,
                 machine_last_part, new_color, runner_lock, plan):
    ranked = []
    for m in machines_to_try:
        # FIX: Check 3-part machine limit
        if not machine_has_capacity_for_new_part(m, part, plan):
            continue
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        if last is None or last == part:
            co_hrs      = 0.0
            color_bonus = 0.0
        else:
            base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
            same_color = (last_color == new_color and last_color not in ("UNKNOWN", "NEEDS_PURGE") and new_color not in ("UNKNOWN",))
            purge      = 0.0 if same_color else (COLOR_PURGE_HRS if last_color not in ("UNKNOWN",) and new_color not in ("UNKNOWN",) else 0.0)
            co_hrs      = base_co + purge
            color_bonus = -0.08 if same_color else 0.0
        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue
        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0
        cost            = count_score + co_penalty + util_penalty + same_part_bonus + color_bonus
        ranked.append((m, co_hrs, effective_free, cost))
    ranked.sort(key=lambda x: x[3])
    return ranked

_phase_a_machines: set = set()

# =============================================================
# SECTION 14B — HELPERS
# =============================================================

def _get_part_total_qty(part, plan):
    return sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == part)

def _current_co_count(plan):
    return sum(1 for r in plan if r.get("Changeover") == "Yes")

def _make_plan_row(part, machine, run_hrs, co_hrs, qty, scenario_id,
                   type_label, role_label, runner_lock=False, phase=1):
    daily    = effective_daily(part)
    monthly  = effective_monthly(part)
    r_val    = rate.get(part, 1)
    inv_now  = inventory.get(part, 0)
    color    = part_color.get(part, "UNKNOWN")
    fixed_m  = part_fixed_machine.get(part)
    last     = machine_state.get(machine)
    l_color  = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
    has_purge = (last is not None and last != part and color != l_color
                 and color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))
    fixed_used = "YES" if (fixed_m and machine == fixed_m) else ("FALLBACK" if fixed_m else "N/A")
    dem_d = demand_daily_raw.get(part, 0.0)
    indent_met_flag = (
        "YES" if (inv_now + qty) >= (daily - 0.5)
        else f"NO — need {daily:.2f}/d, have {inv_now+qty:.0f} pcs"
    )
    demand_met_flag = "YES" if is_demand_met(part, inv_now, qty) else f"NO — need {dem_d:.2f}/d"
    _, t_reason, t_relaxed = terminal_blocked(part)
    return {
        "Part": part,
        "Color": color,
        "Category": part_category.get(part, "Stranger"),
        "Fixed_Machine": fixed_m or "—",
        "Fixed_Used": fixed_used,
        "Machine": machine,
        "Run_Hours": round(run_hrs, 3),
        "Changeover_Hrs": round(co_hrs, 3),
        "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour": round(r_val, 2),
        "Production_Qty": qty,
        "Inventory_Before": round(inv_now, 0),
        "Demand_Today_Required": round(max(0.0, dem_d - inv_now), 2),
        "Demand_Daily": round(dem_d, 2),
        "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
        "Effective_Daily": round(daily, 2),
        "Demand_Driver": demand_driver(part),
        "Monthly_Indent": round(monthly, 0),
        "Today_Target": round(today_target_qty.get(part, 0), 0),
        "Changeover": "No" if co_hrs == 0 else "Yes",
        "Color_Purge": "Yes" if has_purge else "No",
        "Terminal_Relaxed": "YES" if t_relaxed else "No",
        "Terminal_Note": t_reason if t_relaxed else "—",
        "Type": type_label,
        "Role": role_label,
        "Tools_Available": tools_available.get(part, 1),
        "Tools_Used": 1,
        "Runner_Lock": "YES" if runner_lock else "No",
        "Priority_Score": 0,
        "Phase": phase,
        "Indent_Met": indent_met_flag,
        "Demand_Met": demand_met_flag,
        "Stagger_Adjusted": "No",
    }

# =============================================================
# SECTION 14C — INTRA-MACHINE CO RESEQUENCING
# =============================================================

def resequence_machine_rows(plan, machine_last_part_yesterday):
    machine_rows = defaultdict(list)
    other_rows   = []
    for row in plan:
        m = row.get("Machine")
        if m in vt_machines:
            machine_rows[m].append(row)
        else:
            other_rows.append(row)
    resequenced_plan = []
    for m in vt_machines:
        rows = machine_rows.get(m, [])
        if len(rows) <= 1:
            resequenced_plan.extend(rows)
            continue
        yesterday_part = machine_last_part_yesterday.get(m)
        ordered   = []
        remaining = list(rows)
        seed = None
        if yesterday_part:
            for r in remaining:
                if r["Part"] == yesterday_part:
                    seed = r
                    break
        if seed is None:
            seed = max(remaining, key=lambda r: float(r.get("Priority_Score", 0) or 0))
        ordered.append(seed)
        remaining.remove(seed)
        while remaining:
            last_part      = ordered[-1]["Part"]
            last_color_val = part_color.get(last_part, "UNKNOWN")
            def _co_cost(r, _lc=last_color_val):
                p = r["Part"]
                c = part_color.get(p, "UNKNOWN")
                if p == last_part:
                    return -1.0
                base  = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                purge = (COLOR_PURGE_HRS if _lc not in ("UNKNOWN", "NEEDS_PURGE") and c not in ("UNKNOWN",) and _lc != c else 0.0)
                return base + purge
            remaining.sort(key=_co_cost)
            ordered.append(remaining.pop(0))
        for i, row in enumerate(ordered):
            p = row["Part"]
            prev_part = yesterday_part if i == 0 else ordered[i - 1]["Part"]
            if prev_part is None or prev_part == p:
                new_co = 0.0
            else:
                base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                prev_color = ALL_KNOWN_COLORS.get(prev_part, "UNKNOWN")
                new_color  = part_color.get(p, "UNKNOWN")
                purge = (COLOR_PURGE_HRS if prev_color not in ("UNKNOWN", "NEEDS_PURGE") and new_color not in ("UNKNOWN",) and prev_color != new_color else 0.0)
                new_co = base_co + purge
            row["Changeover_Hrs"]  = round(new_co, 3)
            row["Changeover"]      = "No" if new_co == 0 else "Yes"
            row["Total_Hrs_Used"]  = round(new_co + float(row.get("Run_Hours", 0) or 0), 3)
            row["Color_Purge"] = "Yes" if new_co > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001 else "No"
        resequenced_plan.extend(ordered)
    resequenced_plan.extend(other_rows)
    return resequenced_plan

# =============================================================
# SECTION 14D — MACHINE HOURS RECONCILER
# =============================================================

def reconcile_machine_hours(plan, machine_hours):
    recomputed = {m: 0.0 for m in vt_machines}
    for row in plan:
        m = row.get("Machine")
        if m in recomputed:
            recomputed[m] += float(row.get("Run_Hours", 0) or 0) + float(row.get("Changeover_Hrs", 0) or 0)
    for m in vt_machines:
        machine_hours[m] = round(recomputed[m], 4)

# =============================================================
# SECTION 15 — DEMAND-FIRST ASSIGNMENT CORE
# =============================================================

def assign_demand_for_part(part, scenario_id, machine_hours, machine_last_part,
                            current_inventory, plan, already_planned, priority_scores):
    """
    FIX — CORE CHANGE: This function ONLY satisfies demand (production to
    cover today's demand gap). It runs BEFORE any inventory-building pass.
    It respects the 3-part/machine limit but will pick any available machine
    even if it means a changeover.
    """
    daily      = effective_daily(part)
    r_val      = rate.get(part, 1)
    inv_now    = current_inventory.get(part, 0)
    inv_before = inventory.get(part, 0)
    dem_d      = demand_daily_raw.get(part, 0.0)
    dem_gap    = max(0.0, dem_d - inv_now)

    if dem_gap <= 0:
        return []  # Demand already covered by stock

    compatible = vt_compat.get(part, [])
    if not compatible:
        return []

    fixed_m = part_fixed_machine.get(part)
    _, t_note, t_relaxed = terminal_blocked(part)

    hrs_needed = max(MIN_RUN_HOURS, dem_gap / r_val if r_val > 0 else MIN_RUN_HOURS)

    # Fixed-machine parts ONLY run on their designated machine — no fallback to others
    if fixed_m:
        ordered = [fixed_m]
    else:
        # Non-fixed parts: only use machines that are NOT reserved for fixed parts
        ordered = [m for m in compatible if m not in machine_fixed_parts]

    new_rows = []
    produced = 0.0

    for m in ordered:
        if produced >= dem_gap - 0.5:
            break
        if not machine_has_capacity_for_new_part(m, part, plan):
            continue
        # Skip Phase-A fixed machines unless this IS the fixed machine
        if m in _phase_a_machines and m != fixed_m:
            continue
        co_hrs = _co_hrs_for(part, m, machine_last_part)
        # Hard CO cap: bypass only for true zero-inventory stockouts
        if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
            if inv_now > 0:
                continue  # Has some stock — respect cap
            # inv_now == 0: allow CO to prevent total stockout
        free   = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        eff    = round(free - co_hrs, 4)
        if eff < MIN_RUN_HOURS:
            continue

        remain_dem = dem_gap - produced
        run_hrs = max(MIN_RUN_HOURS, min(eff, remain_dem / r_val if r_val > 0 else MIN_RUN_HOURS))
        qty     = round(run_hrs * r_val, 0)

        machine_hours[m]            = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
        current_inventory[part]     = round(current_inventory.get(part, 0) + qty, 0)
        machine_last_part[m]        = part
        produced                   += qty

        last     = machine_state.get(m)
        l_color  = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
        p_color  = part_color.get(part, "UNKNOWN")
        has_purge = (last is not None and last != part and p_color != l_color
                     and p_color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))

        new_rows.append({
            "Part": part,
            "Color": p_color,
            "Category": part_category.get(part, "Stranger"),
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if (fixed_m and m == fixed_m) else ("FALLBACK" if fixed_m else "N/A"),
            "Machine": m,
            "Run_Hours": round(run_hrs, 3),
            "Changeover_Hrs": round(co_hrs, 3),
            "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
            "Rate_Per_Hour": round(r_val, 2),
            "Production_Qty": qty,
            "Inventory_Before": round(inv_before, 0),
            "Demand_Today_Required": round(dem_gap, 2),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Monthly_Indent": round(effective_monthly(part), 0),
            "Today_Target": round(today_target_qty.get(part, 0), 0),
            "Changeover": "No" if co_hrs == 0 else "Yes",
            "Color_Purge": "Yes" if has_purge else "No",
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_note if t_relaxed else "—",
            "Type": "DEMAND-FIRST" + (" [TERMINAL-RELAXED]" if t_relaxed else ""),
            "Role": "Primary",
            "Tools_Available": tools_available.get(part, 1),
            "Tools_Used": 1,
            "Runner_Lock": "No",
            "Priority_Score": priority_scores.get(part, 0),
            "Phase": 0,  # Phase 0 = demand satisfaction
            "Indent_Met": "YES" if (inv_before + produced) >= (daily - 0.5) else f"NO — need {daily:.2f}/d",
            "Demand_Met": "YES" if (inv_before + produced) >= (dem_d - 0.5) else f"NO — still need {max(0,dem_d-(inv_before+produced)):.0f} pcs",
            "Stagger_Adjusted": "No",
        })

    if new_rows:
        already_planned.add(part)

    return new_rows

# =============================================================
# SECTION 16 — INVENTORY BUILD ASSIGNMENT
# =============================================================

def assign_inventory_build(part, scenario_id, machine_hours, machine_last_part,
                            current_inventory, plan, already_planned, priority_scores):
    """
    Runs AFTER demand pass. Builds inventory up to OPD cap.
    Will NOT reduce production of a part that has unmet demand.
    FIX: Respects 3-part machine limit.
    """
    daily      = effective_daily(part)
    r_val      = rate.get(part, 1)
    inv_now    = current_inventory.get(part, 0)
    inv_before = inventory.get(part, 0)
    compatible = vt_compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)
    _, t_note, t_relaxed = terminal_blocked(part)

    if not compatible:
        return []

    # Fixed-machine parts ONLY run on their designated machine
    # Non-fixed parts NEVER run on a fixed machine
    if fixed_m:
        machines_to_rank = [fixed_m] if fixed_m in compatible else []
    else:
        machines_to_rank = [m for m in compatible if m not in machine_fixed_parts]

    if not machines_to_rank:
        return []

    inv_days = inv_now / daily if daily > 0 else 999
    total_shortfall = max(0.0, daily - inv_now)
    hrs_for_full    = max(MIN_RUN_HOURS, total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)

    new_rows      = []
    produced      = 0.0
    tools_used    = 0
    used_machines = set()

    # If part was already assigned (demand pass), tally what's produced
    for r in plan:
        if r["Part"] == part:
            produced   += float(r.get("Production_Qty", 0))
            used_machines.add(r["Machine"])
            tools_used += 1

    if produced >= total_shortfall - 0.5:
        # Demand + indent already met — try OPD build on existing machine
        _do_inv_build_on_existing(part, plan, scenario_id, machine_hours,
                                   current_inventory, r_val, daily, inv_now)
        return []

    # Need more production — find machines
    ranked, runner_lock = rank_machines(
        part, machines_to_rank, machine_hours, machine_last_part, inv_days, plan
    )

    for m, co, eff, _ in ranked:
        if produced >= total_shortfall - 0.5:
            break
        if m in used_machines:
            continue  # already assigned this machine for this part today
        if m in _phase_a_machines and m != fixed_m:
            continue
        # Hard CO cap for inventory-build pass — no exceptions
        if co > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
            continue
        remain = total_shortfall - produced
        run_hrs = max(MIN_RUN_HOURS, min(eff, remain / r_val if r_val > 0 else MIN_RUN_HOURS))
        qty     = round(run_hrs * r_val, 0)

        machine_hours[m]        = round(machine_hours.get(m, 0) + co + run_hrs, 4)
        current_inventory[part] = round(current_inventory.get(part, 0) + qty, 0)
        machine_last_part[m]    = part
        produced               += qty
        tools_used             += 1
        used_machines.add(m)

        last     = machine_state.get(m)
        l_color  = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
        p_color  = part_color.get(part, "UNKNOWN")
        has_purge = (last is not None and last != part and p_color != l_color
                     and p_color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))
        fixed_used = "YES" if (fixed_m and m == fixed_m) else ("FALLBACK" if fixed_m else "N/A")
        indent_met_flag = (
            "YES" if (inv_before + produced) >= (daily - 0.5)
            else f"NO — need {daily:.2f}/d"
        )
        dem_d = demand_daily_raw.get(part, 0.0)
        demand_met_flag = "YES" if is_demand_met(part, inv_before, produced) else f"NO — need {dem_d:.2f}/d"

        new_rows.append({
            "Part": part,
            "Color": p_color,
            "Category": part_category.get(part, "Stranger"),
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": fixed_used,
            "Machine": m,
            "Run_Hours": round(run_hrs, 3),
            "Changeover_Hrs": round(co, 3),
            "Total_Hrs_Used": round(co + run_hrs, 3),
            "Rate_Per_Hour": round(r_val, 2),
            "Production_Qty": qty,
            "Inventory_Before": round(inv_before, 0),
            "Demand_Today_Required": round(max(0.0, dem_d - inv_before), 2),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Monthly_Indent": round(effective_monthly(part), 0),
            "Today_Target": round(today_target_qty.get(part, 0), 0),
            "Changeover": "No" if co == 0 else "Yes",
            "Color_Purge": "Yes" if has_purge else "No",
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_note if t_relaxed else "—",
            "Type": "Indent-Build" + (" [TERMINAL-RELAXED]" if t_relaxed else ""),
            "Role": f"Tool-Expansion (tool {tools_used})" if tools_used > 1 else "Primary",
            "Tools_Available": tools_available.get(part, 1),
            "Tools_Used": tools_used,
            "Runner_Lock": "YES" if runner_lock else "No",
            "Priority_Score": priority_scores.get(part, 0),
            "Phase": 1,
            "Indent_Met": indent_met_flag,
            "Demand_Met": demand_met_flag,
            "Stagger_Adjusted": "No",
        })

    if new_rows:
        already_planned.add(part)
        plan.extend(new_rows)
        # After placing, try OPD build
        _do_inv_build_on_existing(part, plan, scenario_id, machine_hours,
                                   current_inventory, r_val, daily, current_inventory.get(part, 0))

    return new_rows


def _do_inv_build_on_existing(part, plan, scenario_id, machine_hours,
                               current_inventory, r_val, daily, inv_after):
    """Extend the first existing plan row for this part to reach OPD cap."""
    cap_qty    = effective_opd_cap_qty(part, scenario_id, inv_after)
    headroom   = max(0.0, cap_qty - inv_after)
    if headroom <= 0:
        return

    target_row = next((r for r in plan if r["Part"] == part), None)
    if target_row is None:
        return

    m = target_row["Machine"]
    if m in _phase_a_machines:
        return

    free_m = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
    if free_m < 0.05:
        return

    extend_hrs = min(free_m, headroom / r_val if r_val > 0 else 0)
    if extend_hrs < 0.05:
        return

    extra_qty = round(extend_hrs * r_val, 0)
    target_row["Run_Hours"]     = round(float(target_row["Run_Hours"]) + extend_hrs, 3)
    target_row["Total_Hrs_Used"] = round(float(target_row["Changeover_Hrs"]) + float(target_row["Run_Hours"]), 3)
    target_row["Production_Qty"] = round(float(target_row["Production_Qty"]) + extra_qty, 0)
    target_row["Type"] = str(target_row["Type"]) + "+OPD-Build"
    machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)

# =============================================================
# SECTION 17 — FIXED MACHINE SCHEDULING PASS
# =============================================================

def fixed_machine_phase(machine, current_inventory):
    fixed_parts = machine_fixed_parts.get(machine, [])
    if not fixed_parts:
        return "B"
    for p in fixed_parts:
        daily = effective_daily(p)
        inv   = current_inventory.get(p, 0)
        if daily > 0 and inv < SAFETY_DAYS * daily:
            return "A"
    return "B"

def pick_fixed_part_for_today(machine, current_inventory, machine_last_part):
    fixed_parts = machine_fixed_parts.get(machine, [])
    candidates  = []
    for p in fixed_parts:
        daily = effective_daily(p)
        r_val = rate.get(p, 1)
        inv   = current_inventory.get(p, 0)
        if daily <= 0:
            continue
        days_cov     = inv / daily
        hours_needed = daily / r_val if r_val > 0 else 0
        candidates.append((p, days_cov, hours_needed))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (round(x[1], 4), -x[2]))
    if len(candidates) >= 2:
        top, second = candidates[0], candidates[1]
        if abs(top[1] - second[1]) < 0.01:
            last_ran = machine_last_part.get(machine)
            if last_ran == top[0]:
                return second[0]
    return candidates[0][0]

def fixed_part_run_hours(part, phase):
    daily = effective_daily(part)
    r_val = rate.get(part, 1)
    if daily <= 0 or r_val <= 0:
        return AVAILABLE_HOURS
    indent_hrs = daily / r_val
    if phase == "A":
        return AVAILABLE_HOURS_EXTENDED if indent_hrs > 20.0 else AVAILABLE_HOURS
    else:
        return max(MIN_RUN_HOURS, indent_hrs)

def schedule_fixed_machines(machine_hours, machine_last_part,
                              current_inventory, plan, already_planned,
                              priority_scores, scenario_id):
    print(f"\n{'─'*65}")
    print(f"  FIXED MACHINE SCHEDULING PASS")
    print(f"{'─'*65}")

    phase_a_machines = set()
    fixed_plan_rows  = []

    for machine, fixed_parts in sorted(machine_fixed_parts.items()):
        phase   = fixed_machine_phase(machine, current_inventory)
        if phase == "A":
            phase_a_machines.add(machine)
            chosen = pick_fixed_part_for_today(machine, current_inventory, machine_last_part)
            if chosen is None:
                continue
            daily    = effective_daily(chosen)
            r_val    = rate.get(chosen, 1)
            monthly  = effective_monthly(chosen)
            score    = priority_scores.get(chosen, 0)
            inv_before_chosen = inventory.get(chosen, 0)
            run_hrs_cap = fixed_part_run_hours(chosen, "A")
            qty = round(run_hrs_cap * r_val, 0)
            machine_hours[machine]      = round(run_hrs_cap, 4)
            current_inventory[chosen]   = round(current_inventory.get(chosen, 0) + qty, 0)
            machine_last_part[machine]  = chosen
            already_planned.add(chosen)
            _, t_note, t_relaxed = terminal_blocked(chosen)
            dem_d = demand_daily_raw.get(chosen, 0.0)
            row = _make_plan_row(chosen, machine, run_hrs_cap, 0.0, qty, scenario_id,
                                 f"Fixed-PhaseA [{run_hrs_cap}h cap]", "Primary", phase=1)
            row["Priority_Score"] = score
            row["Inventory_Before"] = round(inv_before_chosen, 0)
            plan.append(row)
            fixed_plan_rows.append(row)
            print(f"    Phase A -> {chosen}  {run_hrs_cap:.2f}h  qty={qty:.0f}  driver={demand_driver(chosen)}")
        else:
            for chosen in fixed_parts:
                if machine_hours.get(machine, 0) >= AVAILABLE_HOURS - 0.05:
                    already_planned.add(chosen)
                    continue
                daily   = effective_daily(chosen)
                r_val   = rate.get(chosen, 1)
                score   = priority_scores.get(chosen, 0)
                inv_before_chosen = inventory.get(chosen, 0)
                min_run_for_indent = fixed_part_run_hours(chosen, "B")
                available_now = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
                effective_run = min(min_run_for_indent, available_now)
                if effective_run < MIN_RUN_HOURS:
                    already_planned.add(chosen)
                    continue
                inv_now_chosen = current_inventory.get(chosen, 0)
                cap_qty  = effective_opd_cap_qty(chosen, scenario_id, inv_now_chosen)
                headroom = max(0.0, cap_qty - inv_now_chosen)
                if headroom > 0 and r_val > 0:
                    max_hrs_for_opd = headroom / r_val
                    effective_run = min(available_now, max(effective_run, max_hrs_for_opd))
                    effective_run = max(effective_run, MIN_RUN_HOURS)
                qty = round(effective_run * r_val, 0)
                machine_hours[machine] = round(machine_hours.get(machine, 0) + effective_run, 4)
                current_inventory[chosen] = round(current_inventory.get(chosen, 0) + qty, 0)
                machine_last_part[machine] = chosen
                already_planned.add(chosen)
                row = _make_plan_row(chosen, machine, effective_run, 0.0, qty, scenario_id,
                                     "Fixed-PhaseB", "Primary", phase=1)
                row["Priority_Score"] = score
                row["Inventory_Before"] = round(inv_before_chosen, 0)
                plan.append(row)
                fixed_plan_rows.append(row)
                print(f"    Phase B -> {chosen}  {effective_run:.2f}h  qty={qty:.0f}  driver={demand_driver(chosen)}")

    print(f"\n  Fixed machine pass: {len(phase_a_machines)} Phase A  |  "
          f"{len(machine_fixed_parts) - len(phase_a_machines)} Phase B")
    return phase_a_machines, fixed_plan_rows

# =============================================================
# SECTION 18 — UTILIZATION ENFORCER (DEMAND-SAFE)
# =============================================================

def _is_demand_protected(part, current_inventory):
    """Returns True if this part has unmet demand — NEVER reduce its production."""
    dem = demand_daily_raw.get(part, 0.0)
    if dem <= 0:
        return False
    inv_b = inventory.get(part, 0)
    produced = 0.0  # This would be checked against current plan, simplified here
    return inv_b < dem

def _extend_existing_safe(m, plan, machine_hours, current_inventory,
                           priority_scores, ceiling_days, remaining):
    """Extend existing rows but NEVER at the cost of a demand-protected part."""
    consumed = 0.0
    parts_on_m = sorted(
        [row for row in plan if row["Machine"] == m],
        key=lambda r: float(priority_scores.get(r["Part"], 0) or 0),
        reverse=True,
    )
    for row in parts_on_m:
        if remaining < 0.001:
            break
        p_ext   = row["Part"]
        if is_hard_skip(p_ext):
            continue
        r_ext   = rate.get(p_ext, 1)
        daily_p = effective_daily(p_ext)
        inv_now = current_inventory.get(p_ext, 0)
        headroom = max(0.0, ceiling_days * daily_p - inv_now) if daily_p > 0 else 0
        ext_hrs  = min(remaining, headroom / r_ext if r_ext > 0 else 0)
        if ext_hrs < 0.001:
            continue
        extra_qty = round(ext_hrs * r_ext, 0)
        row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
        row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
        row["Total_Hrs_Used"] = round(float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
        row["Type"] = str(row.get("Type", "")) + f"+Ext{ceiling_days}d"
        machine_hours[m]      = round(machine_hours.get(m, 0) + ext_hrs, 4)
        current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
        remaining = round(remaining - ext_hrs, 4)
        consumed += ext_hrs
    return consumed, remaining


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id, priority_scores):
    print(f"\n  UTILIZATION ENFORCER (demand-protected)")
    floor_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)

    machines_by_util = sorted(vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        if m in _phase_a_machines:
            continue

        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"

        # Try to extend existing parts first
        current_util = machine_hours.get(m, 0)
        if current_util < floor_hrs:
            needed = round(floor_hrs - current_util, 4)
            _, remaining = _extend_existing_safe(
                m, plan, machine_hours, current_inventory,
                priority_scores, opd_cap(scenario_id), min(needed, remaining))
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)

        if remaining < 0.05:
            continue

        # Add new unplanned compatible parts (respecting 3-part limit)
        if remaining >= MIN_RUN_HOURS:
            machine_compat = machine_compatible_parts.get(m, [])
            unplanned = []
            for p in machine_compat:
                if rate.get(p, 0) <= 0:
                    continue
                if indent_monthly.get(p, 0) <= 0 and demand_daily_raw.get(p, 0) <= 0:
                    continue
                t_blk, _, _ = terminal_blocked(p)
                if t_blk:
                    continue
                pfm = part_fixed_machine.get(p)
                if pfm is not None and pfm != m:
                    continue  # Fixed part assigned to a different machine
                if pfm is None and m in machine_fixed_parts:
                    continue  # Non-fixed part trying to use a fixed machine — not allowed
                # FIX: Check 3-part limit
                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                daily_p = effective_daily(p)
                inv_now = current_inventory.get(p, 0)
                cap_q   = effective_opd_cap_qty(p, scenario_id, inv_now)
                if daily_p > 0 and inv_now >= cap_q:
                    continue
                unplanned.append(p)

            def _sort_key(p):
                total_qty_today = _get_part_total_qty(p, plan)
                is_unplanned    = 0 if total_qty_today == 0 else 1
                needs_co        = 0 if (last_on_m is None or last_on_m == p) else 1
                p_col           = part_color.get(p, "UNKNOWN")
                same_col        = 0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN") else 1
                cat_pri         = {"Runner": 0, "Repeater": 1, "Stranger": 2}.get(part_category.get(p, "Stranger"), 2)
                inv_now_p       = current_inventory.get(p, 0)  # lowest inventory first
                return (is_unplanned, needs_co, same_col, cat_pri, inv_now_p)

            unplanned.sort(key=_sort_key)

            for p in unplanned:
                if remaining < MIN_RUN_HOURS:
                    break
                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                co_hrs   = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                    continue
                daily_p   = effective_daily(p)
                r_val     = rate.get(p, 1)
                inv_now_p = current_inventory.get(p, 0)
                cap_qty   = effective_opd_cap_qty(p, scenario_id, inv_now_p)
                headroom  = max(0.0, cap_qty - inv_now_p)
                if headroom <= 0:
                    continue
                shortfall = max(0.0, daily_p - inv_now_p)
                min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
                run_hrs   = max(min_run, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
                qty       = round(run_hrs * r_val, 0)
                inv_before_p = inventory.get(p, 0)
                dem_d = demand_daily_raw.get(p, 0.0)
                _, t_note_f, t_relaxed_f = terminal_blocked(p)
                machine_hours[m]      = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
                current_inventory[p]  = round(current_inventory.get(p, 0) + qty, 0)
                machine_last_part[m]  = p
                already_planned.add(p)
                remaining = round(remaining - co_hrs - run_hrs, 4)
                last_on_m  = machine_last_part.get(m)
                last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
                plan.append({
                    "Part": p, "Color": part_color.get(p, "UNKNOWN"),
                    "Category": part_category.get(p, "Stranger"),
                    "Fixed_Machine": part_fixed_machine.get(p, "—"),
                    "Fixed_Used": "N/A — filler", "Machine": m,
                    "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                    "Rate_Per_Hour": round(r_val, 2),
                    "Production_Qty": qty,
                    "Inventory_Before": round(inv_before_p, 0),
                    "Demand_Today_Required": round(max(0.0, dem_d - inv_before_p), 2),
                    "Demand_Daily": round(dem_d, 2),
                    "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                    "Effective_Daily": round(daily_p, 2),
                    "Demand_Driver": demand_driver(p),
                    "Monthly_Indent": round(effective_monthly(p), 0),
                    "Today_Target": round(today_target_qty.get(p, 0), 0),
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "No",
                    "Terminal_Relaxed": "YES" if t_relaxed_f else "No",
                    "Terminal_Note": t_note_f if t_relaxed_f else "—",
                    "Type": "Filler",
                    "Role": "Primary",
                    "Tools_Available": tools_available.get(p, 1),
                    "Tools_Used": 1, "Runner_Lock": "No",
                    "Priority_Score": round(priority_scores.get(p, 0), 2),
                    "Phase": 2, "Stagger_Adjusted": "No",
                    "Indent_Met": "YES" if (inv_before_p + qty) >= (daily_p - 0.5) else f"NO",
                    "Demand_Met": "YES" if is_demand_met(p, inv_before_p, qty) else "NO",
                })
                print(f"    [FILL] {p:28s} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  driver={demand_driver(p)}")

        # Final extension passes
        for ceiling in [opd_cap(scenario_id), STRATEGIC_BUFFER_DAYS, ABSOLUTE_MAX_DAYS]:
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if remaining < 0.001:
                break
            _, remaining = _extend_existing_safe(
                m, plan, machine_hours, current_inventory,
                priority_scores, ceiling, remaining)


# =============================================================
# SECTION 19 — CO STAGGER
# =============================================================

MIN_CO_GAP_HRS = 20 / 60.0

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours") or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine": m, "part_before": m_rows[i-1]["Part"],
                    "part_after": row["Part"], "co_duration": co_h,
                    "natural_start": cursor, "row_before": m_rows[i-1],
                    "row_after": row,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0) for r in plan if r["Machine"] == m)
    return max(0.0, AVAILABLE_HOURS - used)

def stagger_changeovers(plan, machines, machine_hours):
    print(f"\n  Tool-Changer Serial Queue Scheduler")
    events = _collect_co_events(plan, machines)
    if not events:
        print(f"  No changeovers — tool changer idle")
        return
    events.sort(key=lambda e: e["natural_start"])
    if len(events) > MAX_DAILY_CO:
        events = events[:MAX_DAILY_CO]
    tool_changer_free_at = 0.0
    for idx, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = round(actual_start - natural_start, 4)
        tool_changer_free_at = actual_start + co_h
        # If there's wait time, extend the row before
        if wait_hrs > 0.001:
            rb    = ev["row_before"]
            p_rb  = rb["Part"]
            r_rb  = rate.get(p_rb, 1.0)
            spare = _machine_spare(ev["machine"], plan)
            ext   = min(wait_hrs, spare)
            if ext > 0.001:
                extra = round(ext * r_rb, 0)
                rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + ext, 3)
                rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
                rb["Total_Hrs_Used"] = round(float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
                rb["Stagger_Adjusted"] = f"CO wait +{round(ext*60,1)}min"
                machine_hours[ev["machine"]] = round(machine_hours.get(ev["machine"], 0) + ext, 4)
                current_inventory_ref = {}  # not updated here — handled in reconcile
    print(f"  {len(events)} CO events processed")

# =============================================================
# SECTION 20 — VALIDATION
# =============================================================

def validate_plan_rows(plan, current_inventory):
    violations = []
    for row in plan:
        p     = row["Part"]
        run_h = float(row.get("Run_Hours", 0) or 0)
        v = []
        if run_h < MIN_RUN_HOURS - 0.001:
            v.append(f"Run_Hours={run_h:.3f} < MIN={MIN_RUN_HOURS}")
        if v:
            violations.append({
                "Part": p, "Machine": row.get("Machine", "—"),
                "Run_Hours": run_h,
                "Violations": " | ".join(v),
            })
    print(f"\n  Validation: {len(violations)} violation(s)" if violations else "\n  Validation: all rows OK")
    return violations

# =============================================================
# SECTION 21 — PER-PART DECISION SHEET BUILDER
# =============================================================

def build_part_decision_sheet(all_parts, plan, current_inventory, not_planned_list,
                               deferred_list, priority_scores):
    """
    FIX: New sheet — one row per part explaining:
    - demand today, inventory before, what was produced, why in/out of plan
    - terminal inventory for each required terminal
    - demand met status
    """
    rows = []
    not_planned_reasons = {r["Part"]: r["Reason"] for r in not_planned_list if "Part" in r}
    deferred_reasons    = {r["Part"]: r["Reason"] for r in deferred_list if "Part" in r}

    part_produced = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
        part_machines[row["Part"]].append(row["Machine"])

    for p in sorted(all_parts):
        inv_b    = inventory.get(p, 0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        ind_d    = indent_daily.get(p, 0.0)
        daily    = effective_daily(p)
        r_val    = rate.get(p, None)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        machines  = ", ".join(dict.fromkeys(part_machines[p])) if p in part_machines else "—"
        cat       = part_category.get(p, "Stranger")
        fixed_m   = part_fixed_machine.get(p, "—")
        score     = round(priority_scores.get(p, 0), 2)

        # Demand status
        dem_gap    = max(0.0, dem_d - inv_b)
        dem_covered_by_stock = dem_d > 0 and inv_b >= dem_d
        dem_met    = is_demand_met(p, inv_b, produced)

        # Terminal info
        required_terminals = part_terminals.get(p, [])
        terminal_info_parts = []
        for t in required_terminals:
            t_inv = terminal_status.get(t, None)
            cat_mult = TERMINAL_THRESHOLD.get(cat, 0.25)
            needed   = cat_mult * daily
            status_t = "OK" if (t_inv or 0) >= needed else ("RELAXED" if (t_inv or 0) >= needed * (1 - TERMINAL_RELAXATION_PCT) else "BLOCKED")
            terminal_info_parts.append(f"{t}:inv={t_inv if t_inv is not None else 'MISSING'}:need={needed:.0f}:{status_t}")

        t_blocked, t_reason, t_relaxed = terminal_blocked(p)

        # Determine why in or out of plan
        in_plan = produced > 0
        if in_plan:
            if t_relaxed:
                plan_reason = f"PLANNED (TERMINAL RELAXED) — demand={dem_d:.2f} inv_before={inv_b:.0f}"
            else:
                plan_reason = f"PLANNED — demand={dem_d:.2f} inv_before={inv_b:.0f}"
        elif p in not_planned_reasons:
            plan_reason = f"NOT PLANNED — {not_planned_reasons[p]}"
        elif p in deferred_reasons:
            plan_reason = f"DEFERRED — {deferred_reasons[p]}"
        elif t_blocked:
            plan_reason = f"BLOCKED — {t_reason}"
        elif r_val is None or r_val == 0:
            plan_reason = "SKIPPED — zero/missing cycle time"
        elif daily == 0:
            plan_reason = "SKIPPED — no demand and no indent"
        else:
            skip, skip_reason = should_skip(p)
            if skip:
                plan_reason = f"SKIPPED — {skip_reason}"
            else:
                plan_reason = "NOT SCHEDULED — no machine capacity found"

        rows.append({
            "Part": p,
            "Category": cat,
            "Color": part_color.get(p, "UNKNOWN"),
            "Fixed_Machine": fixed_m,
            "Priority_Score": score,
            "Inventory_Before": round(inv_b, 0),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(ind_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Gap_Today": round(dem_gap, 2),
            "Demand_Covered_By_Stock": "YES" if dem_covered_by_stock else ("NO" if dem_d > 0 else "N/A"),
            "Produced_Today": produced,
            "Inventory_After": inv_after,
            "Machines_Used": machines,
            "Demand_Met": "YES" if dem_met else ("N/A" if dem_d == 0 else "NO"),
            "Indent_Met": (
                "YES" if (inv_after >= daily - 0.5 and daily > 0) else
                ("N/A" if daily == 0 else "NO")
            ),
            "In_Plan": "YES" if in_plan else "NO",
            "Plan_Reason": plan_reason,
            "Terminals_Required": ", ".join(required_terminals) if required_terminals else "None",
            "Terminal_Status_Detail": " | ".join(terminal_info_parts) if terminal_info_parts else "No terminals required",
            "Terminal_Hard_Blocked": "YES" if t_blocked else "No",
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
        })

    return pd.DataFrame(rows)

# =============================================================
# SECTION 22A — PLANNING SUMMARY SHEET
# =============================================================

def build_planning_summary_sheet(plan, not_planned_list, deferred_list,
                                  all_parts, machine_hours, vt_machines,
                                  already_planned_set, scenario_desc):
    """
    Single-sheet executive summary showing:
    - Overall counts: planned / not planned / deferred
    - Total production qty and changeovers
    - Category breakdown (Runner / Repeater / Stranger)
    - Inventory status breakdown after today
    - Demand coverage breakdown
    - Machine utilisation summary
    - Parts NOT planned with reasons
    """
    rows = []

    # ── 1. Overall planning counts ────────────────────────────
    total_parts_universe = len(all_parts)
    parts_in_plan        = len({r["Part"] for r in plan})
    parts_not_planned    = len(not_planned_list)
    parts_deferred       = len(deferred_list)

    total_qty  = sum(float(r.get("Production_Qty", 0)) for r in plan)
    total_co   = sum(1 for r in plan if r.get("Changeover") == "Yes")
    total_co_hrs  = sum(float(r.get("Changeover_Hrs", 0)) for r in plan)
    total_run_hrs = sum(float(r.get("Run_Hours", 0)) for r in plan)

    def _sep(label=""):
        rows.append({"Section": f"── {label} ──", "Metric": "", "Count_or_Value": "", "Detail": ""})

    def _row(section, metric, value, detail=""):
        rows.append({"Section": section, "Metric": metric, "Count_or_Value": value, "Detail": detail})

    _sep("SCENARIO")
    _row("Scenario", "Active Scenario", scenario_desc)

    _sep("PLANNING COUNTS")
    _row("Planning Counts", "Total Parts in Universe",    total_parts_universe)
    _row("Planning Counts", "Parts Successfully Planned", parts_in_plan)
    _row("Planning Counts", "Parts NOT Planned",          parts_not_planned)
    _row("Planning Counts", "Parts Deferred",             parts_deferred)

    _sep("PRODUCTION TOTALS")
    _row("Production Totals", "Total Production Qty (all parts + machines)", round(total_qty, 0))
    _row("Production Totals", "Total Changeovers Today",                     total_co)
    _row("Production Totals", "Total Changeover Hours",                      round(total_co_hrs, 2))
    _row("Production Totals", "Total Run Hours (all machines combined)",     round(total_run_hrs, 2))

    _sep("CATEGORY BREAKDOWN — PLANNED")
    for cat in ("Runner", "Repeater", "Stranger"):
        cat_planned     = len({r["Part"] for r in plan if part_category.get(r["Part"], "Stranger") == cat})
        cat_not_planned = sum(1 for r in not_planned_list
                              if part_category.get(r.get("Part",""), "Stranger") == cat)
        cat_deferred    = sum(1 for r in deferred_list
                              if part_category.get(r.get("Part",""), "Stranger") == cat)
        cat_qty         = sum(float(r.get("Production_Qty", 0)) for r in plan
                              if part_category.get(r["Part"], "Stranger") == cat)
        _row("Category", f"{cat} — Planned",     cat_planned,     f"Qty produced: {round(cat_qty,0):.0f}")
        _row("Category", f"{cat} — Not Planned", cat_not_planned)
        _row("Category", f"{cat} — Deferred",    cat_deferred)

    _sep("INVENTORY STATUS AFTER TODAY")
    part_produced = defaultdict(float)
    for r in plan:
        part_produced[r["Part"]] += float(r.get("Production_Qty", 0))

    status_counts = {"AT_TARGET": 0, "BUILDING": 0, "BELOW_SAFETY": 0, "CRITICAL": 0, "NO_DEMAND": 0}
    for p in all_parts:
        daily    = effective_daily(p)
        inv_b    = inventory.get(p, 0)
        produced = part_produced.get(p, 0)
        inv_after = inv_b + produced
        if daily <= 0:
            status_counts["NO_DEMAND"] += 1
        else:
            days = inv_after / daily
            if days >= TARGET_DAYS:
                status_counts["AT_TARGET"] += 1
            elif days >= SAFETY_DAYS:
                status_counts["BUILDING"] += 1
            elif days >= 1:
                status_counts["BELOW_SAFETY"] += 1
            else:
                status_counts["CRITICAL"] += 1

    _row("Inventory Status", "AT_TARGET  (>= 5 days)",         status_counts["AT_TARGET"])
    _row("Inventory Status", "BUILDING   (>= 3 days, < 5)",    status_counts["BUILDING"])
    _row("Inventory Status", "BELOW_SAFETY (>= 1 day, < 3)",   status_counts["BELOW_SAFETY"])
    _row("Inventory Status", "CRITICAL   (< 1 day or zero)",   status_counts["CRITICAL"])
    _row("Inventory Status", "NO_DEMAND / NO_INDENT (skipped)", status_counts["NO_DEMAND"])

    _sep("DEMAND COVERAGE")
    demand_parts     = [p for p in all_parts if demand_daily_raw.get(p, 0) > 0]
    dem_met_stock    = 0  # covered purely by existing stock
    dem_met_produced = 0  # needed production, now met
    dem_not_met      = 0
    dem_not_planned_parts = []
    for p in demand_parts:
        inv_b    = inventory.get(p, 0)
        produced = part_produced.get(p, 0)
        dem_d    = demand_daily_raw.get(p, 0)
        if inv_b >= dem_d:
            dem_met_stock += 1
        elif is_demand_met(p, inv_b, produced):
            dem_met_produced += 1
        else:
            dem_not_met += 1
            dem_not_planned_parts.append(p)

    _row("Demand Coverage", "Parts with Demand",                       len(demand_parts))
    _row("Demand Coverage", "Demand Met — by existing stock alone",    dem_met_stock)
    _row("Demand Coverage", "Demand Met — by production today",        dem_met_produced)
    _row("Demand Coverage", "Demand NOT Met (shortfall remains)",      dem_not_met,
         ", ".join(dem_not_planned_parts) if dem_not_planned_parts else "—")

    _sep("MACHINE UTILISATION")
    avg_util  = 0.0
    util_list = []
    for m in sorted(vt_machines):
        used     = machine_hours.get(m, 0)
        util_pct = round(used / AVAILABLE_HOURS * 100, 1)
        util_list.append(util_pct)
        parts_on = len({r["Part"] for r in plan if r["Machine"] == m})
        co_on    = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        status   = ("FULL" if used >= AVAILABLE_HOURS - 0.3 else
                    "GOOD" if util_pct >= 98 else
                    "OK"   if util_pct >= 90 else "UNDERUSED")
        _row("Machine Util", m,
             f"{util_pct}%",
             f"Hours={round(used,2)}/{AVAILABLE_HOURS}  Parts={parts_on}/{MAX_PARTS_PER_MACHINE}  CO={co_on}  [{status}]")
    if util_list:
        avg_util = round(sum(util_list) / len(util_list), 1)
    _row("Machine Util", "AVERAGE UTILISATION", f"{avg_util}%")

    _sep("PARTS NOT PLANNED — REASONS")
    if not_planned_list:
        for r in not_planned_list:
            _row("Not Planned", r.get("Part", "?"), "NOT PLANNED", r.get("Reason", "—"))
    else:
        _row("Not Planned", "—", "All eligible parts planned", "")

    _sep("PARTS DEFERRED — REASONS")
    if deferred_list:
        for r in deferred_list:
            _row("Deferred", r.get("Part", "?"), "DEFERRED", r.get("Reason", "—"))
    else:
        _row("Deferred", "—", "No deferred parts", "")

    return pd.DataFrame(rows)


# =============================================================
# SECTION 22 — DAILY TOTALS SHEET
# =============================================================

def build_daily_totals_sheet(plan, machine_hours, vt_machines):
    """
    FIX: New sheet — total production qty, total changeovers, machine-wise summary.
    """
    if not plan:
        return pd.DataFrame()

    total_qty = sum(float(r.get("Production_Qty", 0)) for r in plan)
    total_co  = sum(1 for r in plan if r.get("Changeover") == "Yes")
    total_co_hrs = sum(float(r.get("Changeover_Hrs", 0)) for r in plan)
    total_run_hrs = sum(float(r.get("Run_Hours", 0)) for r in plan)
    parts_scheduled = len({r["Part"] for r in plan})
    machines_active = len({r["Machine"] for r in plan if r.get("Machine") in vt_machines})

    # Demand coverage
    parts_with_demand = [p for p in {r["Part"] for r in plan} if demand_daily_raw.get(p, 0) > 0]
    demand_met_count  = 0
    for p in parts_with_demand:
        inv_b    = inventory.get(p, 0)
        produced = sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == p)
        if is_demand_met(p, inv_b, produced):
            demand_met_count += 1

    rows = [
        {"Metric": "Planning Date", "Value": str(PLANNING_DATE)},
        {"Metric": "Total Production Qty (all parts, all machines)", "Value": round(total_qty, 0)},
        {"Metric": "Total Changeovers Today", "Value": total_co},
        {"Metric": "Total Changeover Hours", "Value": round(total_co_hrs, 2)},
        {"Metric": "Total Run Hours (across all machines)", "Value": round(total_run_hrs, 2)},
        {"Metric": "Parts Scheduled", "Value": parts_scheduled},
        {"Metric": "Machines Active", "Value": machines_active},
        {"Metric": "Parts With Demand", "Value": len(parts_with_demand)},
        {"Metric": "Demand Met (production + stock)", "Value": demand_met_count},
        {"Metric": "Demand NOT Met", "Value": len(parts_with_demand) - demand_met_count},
        {"Metric": "Max Parts Per Machine", "Value": MAX_PARTS_PER_MACHINE},
        {"Metric": "Available Hours Per Machine", "Value": AVAILABLE_HOURS},
        {"Metric": "Min Run Hours", "Value": MIN_RUN_HOURS},
        {"Metric": "", "Value": ""},
        {"Metric": "=== MACHINE-WISE TOTALS ===", "Value": ""},
    ]

    for m in sorted(vt_machines):
        m_rows = [r for r in plan if r["Machine"] == m]
        if not m_rows:
            rows.append({"Metric": f"{m} — IDLE", "Value": 0})
            continue
        m_qty  = sum(float(r.get("Production_Qty", 0)) for r in m_rows)
        m_co   = sum(1 for r in m_rows if r.get("Changeover") == "Yes")
        m_hrs  = machine_hours.get(m, 0)
        m_util = round(m_hrs / AVAILABLE_HOURS * 100, 1)
        m_parts = len({r["Part"] for r in m_rows})
        rows.append({
            "Metric": f"{m} — Parts={m_parts} | CO={m_co} | Util={m_util}% | Hrs={round(m_hrs,2)}",
            "Value": round(m_qty, 0)
        })

    return pd.DataFrame(rows)

# =============================================================
# SECTION 23 — OUTPUT VIEW BUILDERS
# =============================================================

def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])
    rows = []
    for p in sorted(part_qty.keys()):
        daily    = effective_daily(p)
        ind_d    = indent_daily.get(p, 0.0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap = round(produced - daily, 0)
        gap_dir = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        demand_met = is_demand_met(p, inv_b, produced)
        indent_met = (inv_b + produced) >= (ind_d - 0.5) if ind_d > 0 else True
        _, t_note, t_relaxed = terminal_blocked(p)
        rows.append({
            "Part": p, "Category": part_category.get(p, "Stranger"),
            "Machines": ", ".join(dict.fromkeys(part_machines[p])),
            "Total_Qty_Produced": produced,
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Gap_vs_Effective_Daily": gap, "Gap_Direction": gap_dir,
            "Inventory_Before": round(inv_b, 0), "Inventory_After": inv_after,
            "Days_Coverage_After": round(inv_after / daily, 2) if daily > 0 else 0,
            "Demand_Met": "YES" if demand_met else ("N/A" if dem_d == 0 else "NO"),
            "Indent_Met": "YES" if indent_met else "NO",
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Effective_Daily"]).drop(columns=["_sort"]).reset_index(drop=True)
    return df

def build_inventory_target_sheet(plan, all_parts, scenario_id):
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
    rows = []
    for p in sorted(all_parts):
        daily    = effective_daily(p)
        ind_d    = indent_daily.get(p, 0.0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        days_after = round(inv_after / daily, 2) if daily > 0 else 0
        target_qty = round(TARGET_DAYS * daily, 0)
        if inv_after == 0:
            status = "CRITICAL"
        elif days_after < SAFETY_DAYS:
            status = "BELOW_SAFETY"
        elif days_after < TARGET_DAYS:
            status = "BUILDING"
        else:
            status = "AT_TARGET"
        _, t_note, t_relaxed = terminal_blocked(p)
        rows.append({
            "Part": p, "Category": part_category.get(p, "Stranger"),
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Inv_Before": round(inv_b, 0),
            "Produced_Today": produced, "Inv_After": inv_after,
            "Days_Coverage_After": days_after, "Target_Qty_5days": target_qty,
            "Buffer_Status": status,
            "Scheduled_Today": "YES" if produced > 0 else "NO",
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "BUILDING": 2, "AT_TARGET": 3}
        df["_sort"] = df["Buffer_Status"].map(status_order)
        df = df.sort_values(["_sort"], ascending=True).drop(columns=["_sort"]).reset_index(drop=True)
    return df

def build_terminal_status_sheet():
    rows = []
    all_terminals_known = set(terminal_status.keys())
    for terms in part_terminals.values():
        for t in terms:
            all_terminals_known.add(t)
    for t in sorted(all_terminals_known):
        inv = terminal_status.get(t, None)
        parts_needing = [p for p, terms in part_terminals.items() if t in terms]
        parts_hard_blocked, parts_relaxed = [], []
        for p in parts_needing:
            cat   = part_category.get(p, "Stranger")
            daily = effective_daily(p)
            mult  = TERMINAL_THRESHOLD.get(cat, 0.25)
            strict_thresh  = mult * daily
            relaxed_thresh = strict_thresh * (1.0 - TERMINAL_RELAXATION_PCT)
            t_inv = inv if inv is not None else 0
            if t_inv < relaxed_thresh:
                parts_hard_blocked.append(f"{p}(need>={strict_thresh:.0f})")
            elif t_inv < strict_thresh:
                parts_relaxed.append(f"{p}(~{int(TERMINAL_RELAXATION_PCT*100)}%relax)")
        rows.append({
            "Terminal": t,
            "Inventory": round(inv, 0) if inv is not None else "NOT IN SHEET",
            "Parts_Requiring": ", ".join(sorted(parts_needing)) if parts_needing else "—",
            "Parts_Hard_Blocked": ", ".join(parts_hard_blocked) if parts_hard_blocked else "—",
            "Parts_Relaxed": ", ".join(parts_relaxed) if parts_relaxed else "—",
            "Impact": (
                "HARD BLOCKING" if parts_hard_blocked else
                ("RELAXED" if parts_relaxed else
                 ("Adequate" if parts_needing else "No parts"))
            ),
        })
    return pd.DataFrame(rows)

def build_forward_look(current_inventory_after, all_parts):
    rows = []
    for p in all_parts:
        daily = effective_daily(p)
        if daily <= 0:
            continue
        inv_now = current_inventory_after.get(p, 0)
        days_now = inv_now / daily
        days_until_safety = max(0.0, round((inv_now - SAFETY_DAYS * daily) / daily, 1))
        days_until_zero   = max(0.0, round(inv_now / daily, 1))
        alert = ""
        if days_until_zero <= FORWARD_LOOK_DAYS:
            alert = f"ZERO-STOCK RISK in {days_until_zero:.1f} days"
        elif days_until_safety <= FORWARD_LOOK_DAYS:
            alert = f"BELOW SAFETY in {days_until_safety:.1f} days"
        if alert:
            rows.append({
                "Part": p, "Category": part_category.get(p, "Stranger"),
                "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
                "Effective_Daily": round(daily, 2),
                "Inv_After_Today": round(inv_now, 0),
                "Days_Coverage_Today": round(days_now, 2),
                "Days_Until_Safety": days_until_safety,
                "Days_Until_Zero": days_until_zero,
                "Alert": alert,
                "Action": "ESCALATE" if days_until_zero <= 2 else "Plan next 1-3 days",
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("Days_Until_Zero").reset_index(drop=True)
    return df

def build_machine_wise_plan(plan_df, machine_hours):
    if plan_df.empty:
        return pd.DataFrame()
    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue
        for _, pr in machine_rows.iterrows():
            p = pr.get("Part", "—")
            rows.append({
                "Machine": m, "Part": p,
                "Color": part_color.get(p, "UNKNOWN"),
                "Category": part_category.get(p, "Stranger"),
                "Role": pr.get("Role", "Primary"),
                "Run_Hours": round(float(pr.get("Run_Hours", 0) or 0), 2),
                "Changeover_Hrs": round(float(pr.get("Changeover_Hrs", 0) or 0), 3),
                "Production_Qty": round(float(pr.get("Production_Qty", 0) or 0), 0),
                "Rate_Per_Hour": round(float(pr.get("Rate_Per_Hour", 0) or 0), 2),
                "Inventory_Before": round(float(pr.get("Inventory_Before", 0) or 0), 0),
                "Demand_Daily": round(float(pr.get("Demand_Daily", 0) or 0), 2),
                "Demand_Today_Required": round(float(pr.get("Demand_Today_Required", 0) or 0), 2),
                "Demand_Met": pr.get("Demand_Met", "—"),
                "Effective_Daily": round(float(pr.get("Effective_Daily", 0) or 0), 2),
                "Demand_Driver": pr.get("Demand_Driver", "—"),
                "Terminal_Relaxed": pr.get("Terminal_Relaxed", "No"),
                "Type": pr.get("Type", "Primary") or "Primary",
                "Row_Type": "Part",
            })
        co_total  = machine_rows["Changeover_Hrs"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        run_total = machine_rows["Run_Hours"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        qty_total = machine_rows["Production_Qty"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        hrs_total = round(co_total + run_total, 2)
        n_parts   = len(machine_rows["Part"].unique())
        rows.append({
            "Machine": m, "Part": f"TOTAL — {m}  [Parts={n_parts}/{MAX_PARTS_PER_MACHINE}]",
            "Color": "—", "Category": "—", "Role": "—",
            "Run_Hours": round(run_total, 2),
            "Changeover_Hrs": round(co_total, 2),
            "Production_Qty": round(qty_total, 0),
            "Rate_Per_Hour": "—",
            "Inventory_Before": "—",
            "Demand_Daily": "—",
            "Demand_Today_Required": "—",
            "Demand_Met": "—",
            "Effective_Daily": "—",
            "Demand_Driver": "—",
            "Terminal_Relaxed": "—",
            "Type": f"Total {hrs_total}h / {AVAILABLE_HOURS}h  |  Util {round(hrs_total/AVAILABLE_HOURS*100,1)}%",
            "Row_Type": "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})
    return pd.DataFrame(rows)

def build_co_queue(plan, machines):
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()
    events.sort(key=lambda e: e["natural_start"])
    rows = []
    for pos, ev in enumerate(events, 1):
        before_color = part_color.get(ev["part_before"], "UNKNOWN")
        after_color  = part_color.get(ev["part_after"],  "UNKNOWN")
        color_change = before_color != after_color and before_color != "UNKNOWN" and after_color != "UNKNOWN"
        rows.append({
            "Queue_Position": pos, "Machine": ev["machine"],
            "Part_Before": ev["part_before"], "Part_After": ev["part_after"],
            "Color_Before": before_color, "Color_After": after_color,
            "Color_Change": "YES — PURGE" if color_change else "No",
            "CO_Duration_Min": round(ev["co_duration"] * 60, 1),
        })
    return pd.DataFrame(rows)

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv    = inventory.get(p, 0.0)
        monthly = effective_monthly(p)
        daily  = effective_daily(p)
        skip, skip_reason = should_skip(p)
        days_cov = inv / daily if daily > 0 else 0
        _, t_note, t_relaxed = terminal_blocked(p)
        if inv == 0.0 and monthly > 0:
            status = "ZERO INV"
        elif skip and inv >= TARGET_DAYS * daily:
            status = "AT TARGET"
        elif skip:
            status = "SKIPPED"
        else:
            status = "PRODUCTION NEEDED"
        rows.append({
            "Part": p,
            "Indent_Monthly": round(indent_monthly.get(p, 0.0), 0),
            "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Inventory_Now": round(inv, 0),
            "Days_Coverage": round(days_cov, 2),
            "Indent_Status": status,
            "Skip_Reason": skip_reason,
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 24 — PART AUDIT
# =============================================================

def build_audit(all_universe_parts, matrix_parts, zero_rate_set):
    audit_rows = []
    for part in all_universe_parts:
        inv    = inventory.get(part, 0.0)
        r_val  = rate.get(part, None)
        monthly = effective_monthly(part)
        ind_d  = indent_daily.get(part, 0.0)
        dem_d  = demand_daily_raw.get(part, 0.0)
        daily  = effective_daily(part)
        days_cov = inv / daily if daily > 0 else 0
        t_blk, t_rsn, t_relaxed = terminal_blocked(part)

        if part in zero_rate_set or r_val is None:
            status = "ZERO/MISSING CYCLE TIME"
        elif part not in matrix_parts:
            status = "NOT IN VT_MATRIX"
        elif monthly == 0:
            status = "ZERO INDENT+DEMAND"
        elif daily <= MIN_DAILY_INDENT and dem_d <= 0:
            status = "SKIPPED (LOW INDENT, NO DEMAND)"
        elif daily > 0 and inv >= TARGET_DAYS * daily:
            status = "AT TARGET — SKIP"
        elif t_blk:
            status = "BLOCKED — TERMINAL (HARD)"
        elif t_relaxed:
            status = "ENTERS SCHEDULER [TERMINAL RELAXED]"
        else:
            status = "ENTERS SCHEDULER"

        audit_rows.append({
            "Part": part, "Color": part_color.get(part, "UNKNOWN"),
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Inventory": round(inv, 0), "Days_Coverage": round(days_cov, 2),
            "Rate_Per_Hour": round(r_val, 2) if r_val else "—",
            "Status": status,
        })
    return pd.DataFrame(audit_rows)

# =============================================================
# SECTION 25 — MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):
    global _phase_a_machines
    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
    print(f"  OPD cap: {opd_cap(scenario_id)} days")

    horizon_df = compute_indent_horizon(parts)

    active_parts = [
        p for p in parts
        if not should_skip(p)[0] and effective_daily(p) > 0
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_hours      = {m: 0.0 for m in vt_machines}
    machine_last_part  = {m: machine_state.get(m) for m in vt_machines}
    current_inventory  = inventory.copy()

    plan            = []
    already_planned = set()
    not_planned     = []
    deferred        = []

    # ── PASS 0: Fixed machines ────────────────────────────────
    _phase_a_machines, _ = schedule_fixed_machines(
        machine_hours, machine_last_part, current_inventory,
        plan, already_planned, priority_scores, scenario_id)

    # ── PASS 1: DEMAND-FIRST — all parts with uncovered demand
    # Sort: demand gap largest first (most urgent first)
    # FIX: Parts with demand > inventory are ALWAYS scheduled first
    demand_critical = [
        p for p in active_parts
        if p not in already_planned
        and demand_daily_raw.get(p, 0) > 0
        and current_inventory.get(p, 0) < demand_daily_raw.get(p, 0)
        and vt_compat.get(p)
    ]
    demand_critical.sort(
        key=lambda p: (
            -(demand_daily_raw.get(p, 0) - current_inventory.get(p, 0)),  # largest gap first
            current_inventory.get(p, 0),  # lowest inventory first
        )
    )

    print(f"\n  {'─'*65}")
    print(f"  PASS 1: DEMAND-FIRST  ({len(demand_critical)} parts with unmet demand today)")
    print(f"  {'─'*65}")

    for part in demand_critical:
        t_blocked, t_reason, t_relaxed = terminal_blocked(part)
        if t_blocked:
            not_planned.append({"Part": part, "Reason": t_reason})
            continue
        new_rows = assign_demand_for_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores
        )
        if new_rows:
            plan.extend(new_rows)
            total_qty = sum(float(r["Production_Qty"]) for r in new_rows)
            dem_d = demand_daily_raw.get(part, 0)
            inv_b = inventory.get(part, 0)
            dem_met_str = "✓ DEMAND MET" if is_demand_met(part, inv_b, total_qty) else "✗ PARTIAL"
            print(f"  {part:<30} -> {new_rows[0]['Machine']:<20}  qty={total_qty:.0f}  "
                  f"dem={dem_d:.2f}  inv_b={inv_b:.0f}  {dem_met_str}")
        else:
            not_planned.append({"Part": part, "Reason": "No machine capacity for demand"})
            print(f"  {part:<30} -> NO CAPACITY (demand={demand_daily_raw.get(part,0):.2f}  inv={current_inventory.get(part,0):.0f})")

    # ── PASS 2: INDENT BUILD — remaining active parts by priority
    print(f"\n  {'─'*65}")
    print(f"  PASS 2: INDENT BUILD (remaining {len(active_parts) - len(already_planned)} parts)")
    print(f"  {'─'*65}")

    remaining_parts = sorted(
        [p for p in active_parts if p not in already_planned and vt_compat.get(p)],
        key=lambda p: (
            inventory.get(p, 0),          # lowest absolute inventory first
            -priority_scores.get(p, 0),   # then highest priority score
        )
    )

    for part in remaining_parts:
        monthly = effective_monthly(part)
        if monthly == 0:
            deferred.append({"Part": part, "Reason": "Effective monthly = 0"})
            continue
        t_blocked, t_reason, t_relaxed = terminal_blocked(part)
        if t_blocked:
            not_planned.append({"Part": part, "Reason": t_reason})
            continue
        new_rows = assign_inventory_build(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores
        )
        if new_rows:
            total_qty = sum(float(r["Production_Qty"]) for r in new_rows)
            print(f"  {part:<30} -> {new_rows[0]['Machine']:<20}  qty={total_qty:.0f}  driver={demand_driver(part)}")
        elif part not in already_planned:
            not_planned.append({"Part": part, "Reason": "No capacity (indent pass)"})

    # ── PASS 3: UTILIZATION ENFORCEMENT ──────────────────────
    utilization_enforcer(
        plan, machine_hours, machine_last_part, list(parts),
        already_planned, current_inventory, scenario_id, priority_scores)

    # ── FINALIZE ──────────────────────────────────────────────
    reconcile_machine_hours(plan, machine_hours)
    plan = resequence_machine_rows(plan, machine_state)
    reconcile_machine_hours(plan, machine_hours)

    for m in vt_machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if m_rows:
            machine_last_part[m] = m_rows[-1]["Part"]

    stagger_changeovers(plan, vt_machines, machine_hours)
    reconcile_machine_hours(plan, machine_hours)

    violations = validate_plan_rows(plan, current_inventory)

    # ── Build output DFs ──────────────────────────────────────
    forward_look_df   = build_forward_look(current_inventory, list(parts))
    prod_vs_indent_df = build_production_vs_indent(plan, list(parts))
    inv_target_df     = build_inventory_target_sheet(plan, list(parts), scenario_id)
    violations_df     = pd.DataFrame(violations) if violations else pd.DataFrame()
    daily_totals_df   = build_daily_totals_sheet(plan, machine_hours, vt_machines)

    mach_rows = []
    for m in vt_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        mach_rows.append({
            "Machine": m, "Used_Hours": round(used, 2),
            "Unused_Hours": round(AVAILABLE_HOURS - used, 2),
            "Utilization_%": util_pct,
            "CO_Count": co_count,
            "Parts_Count": len(parts_run),
            "Parts_At_Limit": "YES" if len(parts_run) >= MAX_PARTS_PER_MACHINE else "No",
            "Status": (
                "FULL"     if used >= AVAILABLE_HOURS - 0.3 else
                "GOOD"     if util_pct >= 98 else
                "OK"       if util_pct >= 90 else "UNDERUSED"
            ),
            "Last_Part": machine_last_part.get(m) or "—",
            "All_Parts": ", ".join(parts_run) if parts_run else "— idle —",
        })

    inv_rows = []
    for p in parts:
        ind_d    = indent_daily.get(p, 0.0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        daily    = effective_daily(p)
        inv_b    = inventory.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        demand_ok = is_demand_met(p, inv_b, produced)
        inv_rows.append({
            "Part": p,
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(ind_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Inv_Before": round(inv_b, 0), "Produced_Today": round(produced, 0),
            "Inv_After": round(inv_after, 0), "Days_Coverage": round(days_cov, 2),
            "Demand_Met_Today": "YES" if demand_ok else ("N/A" if dem_d == 0 else "NO"),
            "Status": ("AT_TARGET" if days_cov >= TARGET_DAYS else
                       "OK"        if days_cov >= SAFETY_DAYS else
                       "LOW"       if days_cov >= 1 else "CRITICAL"),
        })

    micro_idle_log = []
    for m in vt_machines:
        used = machine_hours.get(m, 0)
        remaining = round(AVAILABLE_HOURS - used, 4)
        if remaining >= 0.25:
            micro_idle_log.append({
                "Machine": m, "Idle_Hrs": round(remaining, 3),
                "Utilization_Pct": round((1 - remaining / AVAILABLE_HOURS) * 100, 1),
                "Note": f"Parts on machine: {len({r['Part'] for r in plan if r['Machine']==m})}/{MAX_PARTS_PER_MACHINE}",
            })

    plan_df   = pd.DataFrame(plan) if plan else pd.DataFrame()
    def_df    = pd.DataFrame(deferred) if deferred else pd.DataFrame()
    not_df    = pd.DataFrame(not_planned) if not_planned else pd.DataFrame()
    mach_df   = pd.DataFrame(mach_rows)
    inv_df    = pd.DataFrame(inv_rows)
    micro_df  = pd.DataFrame(micro_idle_log) if micro_idle_log else pd.DataFrame()

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario", scenario_desc)

    # Demand coverage summary
    demand_parts = [p for p in parts if demand_daily_raw.get(p, 0) > 0]
    demand_met_count = 0
    demand_not_met   = []
    for p in demand_parts:
        inv_b    = inventory.get(p, 0)
        produced = sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == p)
        if is_demand_met(p, inv_b, produced):
            demand_met_count += 1
        else:
            demand_not_met.append(p)

    total_production = sum(float(r.get("Production_Qty", 0)) for r in plan)
    total_co_count   = sum(1 for r in plan if r.get("Changeover") == "Yes")

    print(f"\n  {'='*65}")
    print(f"  SUMMARY — {scenario_desc}")
    print(f"    Parts planned      : {len(already_planned)}")
    print(f"    Not planned        : {len(not_planned)}")
    print(f"    Deferred           : {len(deferred)}")
    print(f"    Violations         : {len(violations)}")
    print(f"    Total Production   : {total_production:.0f} pcs")
    print(f"    Total Changeovers  : {total_co_count}")
    if demand_parts:
        print(f"    Demand parts       : {len(demand_parts)}")
        print(f"    Demand MET         : {demand_met_count}/{len(demand_parts)}")
        if demand_not_met:
            print(f"    Demand NOT MET     : {', '.join(demand_not_met)}")
    if not mach_df.empty:
        print(f"    Avg utilization    : {mach_df['Utilization_%'].mean():.1f}%")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df, machine_last_part,
            horizon_df, score_df, micro_df,
            prod_vs_indent_df, inv_target_df, forward_look_df,
            violations_df, daily_totals_df, priority_scores)

# =============================================================
# SECTION 26 — PART UNIVERSE SETUP & RUN
# =============================================================

all_book_parts     = list(data["Material"].unique())
all_demand_parts   = list(demand_daily_raw.keys())
all_universe_parts = list(dict.fromkeys(all_book_parts + all_demand_parts))

matrix_parts  = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set = set(data_zero_rate["Material"].unique())

for p in all_demand_parts:
    if p not in inventory:
        inventory[p] = 0.0
    if p not in part_color:
        part_color[p]  = "UNKNOWN"
        ALL_KNOWN_COLORS[p] = "UNKNOWN"

audit_df = build_audit(all_universe_parts, matrix_parts, zero_rate_set)
vt_terminal_status_df = build_terminal_status_sheet()

print(f"\n  Part audit ({len(all_universe_parts)} total):")
for status, count in audit_df["Status"].value_counts().items():
    marker = "+" if "ENTERS SCHEDULER" in status else "·"
    print(f"    {marker}  {status:<60}: {count:>4}")

schedulable_parts = [
    p for p in all_universe_parts
    if p in matrix_parts and (rate.get(p, 0) or 0) > 0
]

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_scores, vt_micro,
 vt_prod_vs_indent, vt_inv_target, vt_forward_look,
 vt_violations, vt_daily_totals, priority_scores_final) = schedule(schedulable_parts, "VT Machines")

save_machine_state(vt_state)

# Build final sheets that need plan + priority scores
vt_mw     = build_machine_wise_plan(vt_plan, {m: 0.0 for m in vt_machines})
vt_co_q   = build_co_queue(
    vt_plan.to_dict("records") if not vt_plan.empty else [],
    vt_machines
)

# Build per-part decision sheet
vt_part_decision = build_part_decision_sheet(
    all_universe_parts,
    vt_plan.to_dict("records") if not vt_plan.empty else [],
    {p: inventory.get(p, 0) for p in all_universe_parts},  # use original inventory for decision sheet
    vt_not.to_dict("records") if not vt_not.empty else [],
    vt_def.to_dict("records") if not vt_def.empty else [],
    priority_scores_final,
)

# Build planning summary sheet
_scenario_id_final, _scenario_desc_final = classify_scenario(schedulable_parts)
_machine_hours_final = {
    m: (float(vt_mach.loc[vt_mach["Machine"] == m, "Used_Hours"].values[0])
        if not vt_mach.empty and m in vt_mach["Machine"].values else 0.0)
    for m in vt_machines
}
vt_planning_summary = build_planning_summary_sheet(
    vt_plan.to_dict("records") if not vt_plan.empty else [],
    vt_not.to_dict("records") if not vt_not.empty else [],
    vt_def.to_dict("records") if not vt_def.empty else [],
    all_universe_parts,
    _machine_hours_final,
    vt_machines,
    set(vt_plan["Part"].unique()) if not vt_plan.empty else set(),
    _scenario_desc_final,
)

# =============================================================
# SECTION 27 — EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Planning_Summary":      "0F3460",
    "VT_Daily_Totals":          "1A1A2E",
    "VT_Plan_By_Machine":       "0D6E6E",
    "VT_CO_Queue":              "375623",
    "VT_Plan":                  "1F4E79",
    "VT_Part_Decision":         "7B2D8B",
    "VT_Production_vs_Indent":  "154360",
    "VT_Inventory_Target":      "1B4F72",
    "VT_Priority_Scores":       "2C4770",
    "VT_Machine_Util":          "375623",
    "VT_Not_Planned":           "7B2C2C",
    "VT_Deferred":              "7F6000",
    "VT_Inventory_Health":      "4A235A",
    "VT_Indent_Horizon":        "154360",
    "VT_Part_Audit":            "1C3557",
    "VT_Micro_Idle":            "5C3D2E",
    "VT_Terminal_Status":       "7B1C1C",
    "VT_Forward_Look":          "6D28D9",
    "VT_Violations":            "991B1B",
}

STATUS_FILLS = {
    "FULL":                                  PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":                                  PatternFill("solid", fgColor="DDEBF7"),
    "OK":                                    PatternFill("solid", fgColor="EBF5E1"),
    "UNDERUSED":                             PatternFill("solid", fgColor="FFC7CE"),
    "AT_TARGET":                             PatternFill("solid", fgColor="C6EFCE"),
    "BUILDING":                              PatternFill("solid", fgColor="DDEBF7"),
    "BELOW_SAFETY":                          PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":                              PatternFill("solid", fgColor="FFC7CE"),
    "LOW":                                   PatternFill("solid", fgColor="FFEB9C"),
    "OVER":                                  PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":                                 PatternFill("solid", fgColor="FFC7CE"),
    "MET":                                   PatternFill("solid", fgColor="C6EFCE"),
    "YES":                                   PatternFill("solid", fgColor="C6EFCE"),
    "NO":                                    PatternFill("solid", fgColor="FFC7CE"),
    "YES — PURGE":                           PatternFill("solid", fgColor="FFC7CE"),
    "HARD BLOCKING":                         PatternFill("solid", fgColor="FFC7CE"),
    "RELAXED":                               PatternFill("solid", fgColor="FFEB9C"),
    "Adequate":                              PatternFill("solid", fgColor="C6EFCE"),
    "ENTERS SCHEDULER":                      PatternFill("solid", fgColor="C6EFCE"),
    "ENTERS SCHEDULER [TERMINAL RELAXED]":   PatternFill("solid", fgColor="FFEB9C"),
    "BLOCKED — TERMINAL (HARD)":            PatternFill("solid", fgColor="FFC7CE"),
    "N/A":                                   PatternFill("solid", fgColor="F2F2F2"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    status_cols = ["Status", "Gap_Direction", "Buffer_Status", "Color_Change",
                   "Impact", "Result", "Demand_Met", "Demand_Met_Today",
                   "Indent_Met", "Demand_Driver", "Terminal_Relaxed",
                   "Demand_Covered_By_Stock", "In_Plan", "Terminal_Hard_Blocked",
                   "Parts_At_Limit"]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in status_cols):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"

def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"), PatternFill("solid", fgColor="F0FDF4")]
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30
    headers       = [cell.value for cell in ws[1]]
    row_type_col  = headers.index("Row_Type") + 1 if "Row_Type" in headers else None
    machine_col   = headers.index("Machine")  + 1 if "Machine"  in headers else None
    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col - 1].value if row_type_col else ""
        machine  = row[machine_col  - 1].value if machine_col  else ""
        if machine and machine != current_machine and str(machine).strip():
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif row_type == "Part":
            for cell in row:
                cell.fill = part_fills[machine_color_idx]
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "B2"

def style_totals_sheet(ws):
    """Special styling for daily totals — larger font, highlights."""
    header_fill = PatternFill("solid", fgColor="1A1A2E")
    key_fill    = PatternFill("solid", fgColor="16213E")
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = Font(bold=True, color="FFFFFF", size=12)
        cell.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 30
    for row in ws.iter_rows(min_row=2):
        metric_cell = row[0]
        value_cell  = row[1] if len(row) > 1 else None
        if metric_cell.value and "===" in str(metric_cell.value):
            for cell in row:
                cell.fill = key_fill
                cell.font = Font(bold=True, color="FFFF00", size=11)
        elif metric_cell.value and any(kw in str(metric_cell.value) for kw in
                                        ["Total Production", "Total Changeovers", "Demand MET", "Demand NOT Met"]):
            metric_cell.font = Font(bold=True, size=12)
            if value_cell:
                value_cell.font = Font(bold=True, size=12, color="005500" if "MET" in str(metric_cell.value) else "000000")
    ws.column_dimensions["A"].width = 60
    ws.column_dimensions["B"].width = 20
    ws.freeze_panes = "A2"

def style_planning_summary_sheet(ws):
    """Style the planning summary — section separators highlighted, data rows alternating."""
    header_fill  = PatternFill("solid", fgColor="0F3460")
    section_fill = PatternFill("solid", fgColor="1A4A7A")
    good_fill    = PatternFill("solid", fgColor="C6EFCE")
    warn_fill    = PatternFill("solid", fgColor="FFEB9C")
    bad_fill     = PatternFill("solid", fgColor="FFC7CE")
    alt_fill     = PatternFill("solid", fgColor="F0F4FF")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers = [cell.value for cell in ws[1]]
    section_col_idx = headers.index("Section") + 1 if "Section" in headers else 1
    metric_col_idx  = headers.index("Metric")  + 1 if "Metric"  in headers else 2
    value_col_idx   = headers.index("Count_or_Value") + 1 if "Count_or_Value" in headers else 3

    alt = False
    for row in ws.iter_rows(min_row=2):
        section_val = str(row[section_col_idx - 1].value or "")
        value_val   = str(row[value_col_idx - 1].value or "")
        metric_val  = str(row[metric_col_idx - 1].value or "")

        if section_val.startswith("──"):
            for cell in row:
                cell.fill = section_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
            alt = False
        else:
            row_fill = alt_fill if alt else PatternFill()
            alt = not alt
            for cell in row:
                cell.fill = row_fill

            # Colour-code value cell
            v_cell = row[value_col_idx - 1]
            if any(kw in metric_val for kw in ("Planned", "Met — by")):
                v_cell.fill = good_fill
                v_cell.font = Font(bold=True, color="006100")
            elif any(kw in metric_val for kw in ("NOT Planned", "NOT Met", "CRITICAL", "NOT Planned")):
                v_cell.fill = bad_fill
                v_cell.font = Font(bold=True, color="9C0006")
            elif any(kw in metric_val for kw in ("Deferred", "BELOW_SAFETY", "BUILDING")):
                v_cell.fill = warn_fill
            elif "AVERAGE UTIL" in metric_val:
                pct = float(value_val.replace("%", "")) if "%" in value_val else 0
                v_cell.fill = good_fill if pct >= 90 else (warn_fill if pct >= 70 else bad_fill)
                v_cell.font = Font(bold=True)

    ws.column_dimensions["A"].width = 22
    ws.column_dimensions["B"].width = 50
    ws.column_dimensions["C"].width = 22
    ws.column_dimensions["D"].width = 55
    ws.freeze_panes = "A2"


print(f"\nWriting output -> {output_path}")

sheets = {
    "VT_Planning_Summary":     vt_planning_summary,
    "VT_Daily_Totals":         vt_daily_totals,
    "VT_Plan_By_Machine":      vt_mw,
    "VT_CO_Queue":             vt_co_q,
    "VT_Plan":                 vt_plan,
    "VT_Part_Decision":        vt_part_decision,
    "VT_Production_vs_Indent": vt_prod_vs_indent,
    "VT_Inventory_Target":     vt_inv_target,
    "VT_Priority_Scores":      vt_scores,
    "VT_Machine_Util":         vt_mach,
    "VT_Not_Planned":          vt_not,
    "VT_Deferred":             vt_def,
    "VT_Inventory_Health":     vt_inv,
    "VT_Indent_Horizon":       vt_horizon,
    "VT_Part_Audit":           audit_df,
    "VT_Terminal_Status":      vt_terminal_status_df,
    "VT_Forward_Look":         vt_forward_look,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro
if not vt_violations.empty:
    sheets["VT_Violations"] = vt_violations

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine" in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])
if "VT_Daily_Totals" in wb.sheetnames:
    style_totals_sheet(wb["VT_Daily_Totals"])
if "VT_Planning_Summary" in wb.sheetnames:
    style_planning_summary_sheet(wb["VT_Planning_Summary"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name not in ("VT_Plan_By_Machine", "VT_Daily_Totals", "VT_Planning_Summary"):
        style_sheet(wb[sheet_name], header_hex)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied")

# =============================================================
# SECTION 28 — FINAL SUMMARY
# =============================================================

total_prod = sum(float(r.get("Production_Qty", 0)) for r in (vt_plan.to_dict("records") if not vt_plan.empty else []))
total_cos  = sum(1 for r in (vt_plan.to_dict("records") if not vt_plan.empty else []) if r.get("Changeover") == "Yes")

print(f"\n{'='*65}")
print(f"  Smart APS V12-DEMAND  —  {PLANNING_DATE}")
print(f"  Max parts/machine: {MAX_PARTS_PER_MACHINE}  |  Min run: {MIN_RUN_HOURS}h  |  Max CO: {MAX_DAILY_CO}")
print(f"{'='*65}")
print(f"\n  *** TODAY'S TOTALS ***")
print(f"    Total Production   : {total_prod:.0f} pcs")
print(f"    Total Changeovers  : {total_cos}")

for status, count in audit_df["Status"].value_counts().items():
    marker = "+" if "ENTERS SCHEDULER" in status else "·"
    print(f"  {marker} {status:<60}: {count:>4}")

print(f"\n  Results:")
print(f"    Plan rows          : {len(vt_plan):>4}")
print(f"    Not planned        : {len(vt_not):>4}")
print(f"    Deferred           : {len(vt_def):>4}")
print(f"    Violations         : {len(vt_violations):>4}")

if not vt_mach.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average : {vt_mach['Utilization_%'].mean():.1f}%")

if not vt_inv_target.empty:
    at_t = (vt_inv_target["Buffer_Status"] == "AT_TARGET").sum()
    bld  = (vt_inv_target["Buffer_Status"] == "BUILDING").sum()
    bls  = (vt_inv_target["Buffer_Status"] == "BELOW_SAFETY").sum()
    crt  = (vt_inv_target["Buffer_Status"] == "CRITICAL").sum()
    print(f"\n  Inventory after today:")
    print(f"    AT_TARGET  : {at_t:>4}")
    print(f"    BUILDING   : {bld:>4}")
    print(f"    BELOW_SAFE : {bls:>4}")
    print(f"    CRITICAL   : {crt:>4}")

if not vt_prod_vs_indent.empty and "Demand_Met" in vt_prod_vs_indent.columns:
    dm_yes = (vt_prod_vs_indent["Demand_Met"] == "YES").sum()
    dm_no  = (vt_prod_vs_indent["Demand_Met"] == "NO").sum()
    print(f"\n  Demand coverage (parts in plan):")
    print(f"    MET     : {dm_yes:>4}")
    print(f"    NOT MET : {dm_no:>4}")

print(f"\n  Output -> {output_path}")
print(f"  State  -> {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date(2026, 4, 10)")
print(f"{'='*65}")